In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
reviews_path = "/content/drive/MyDrive/Colab Notebooks/recommender_data/Electronics_5.json.gz"
meta_path = "/content/drive/MyDrive/Colab Notebooks/recommender_data/meta_Electronics.json.gz"

In [26]:
import os
os.listdir('/content/drive/MyDrive/Colab Notebooks/recommender_data')

['Amazon Electronics Metadata.csv.zip',
 'Electronics_5.json.zip',
 'Electronics_5.json',
 'Amazon Electronics Metadata.csv']

In [27]:
import zipfile

# unzip reviews
with zipfile.ZipFile('/content/drive/MyDrive/Colab Notebooks/recommender_data/Electronics_5.json.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/drive/MyDrive/Colab Notebooks/recommender_data')

# unzip metadata
with zipfile.ZipFile('/content/drive/MyDrive/Colab Notebooks/recommender_data/Amazon Electronics Metadata.csv.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/drive/MyDrive/Colab Notebooks/recommender_data')

In [28]:
import os
os.listdir('/content/drive/MyDrive/Colab Notebooks/recommender_data')

['Amazon Electronics Metadata.csv.zip',
 'Electronics_5.json.zip',
 'Electronics_5.json',
 'Amazon Electronics Metadata.csv']

In [29]:
reviews_path = "/content/drive/MyDrive/Colab Notebooks/recommender_data/Electronics_5.json"
meta_path = "/content/drive/MyDrive/Colab Notebooks/recommender_data/Amazon Electronics Metadata.csv"

In [30]:
import pandas as pd

reviews = pd.read_json(reviews_path, lines=True)

In [31]:
meta = pd.read_csv(meta_path)

In [32]:
# 1. Inspect  & Standardize

In [33]:
# ispect
reviews.columns
meta.columns

Index(['asin', 'imUrl', 'description', 'categories', 'title', 'price',
       'salesRank', 'related', 'brand'],
      dtype='object')

In [34]:
reviews.head(2)
meta.head(2)

,asin,imUrl,description,categories,title,price,salesRank,related,brand
0,0132793040,http://ecx.images-amazon.com/images/I/31JIPhp%...,The Kelby Training DVD Mastering Blend Modes i...,"[['Electronics', 'Computers & Accessories', 'C...",Kelby Training DVD: Mastering Blend Modes in A...,NaN,NaN,NaN,NaN
1,0321732944,http://ecx.images-amazon.com/images/I/31uogm6Y...,NaN,"[['Electronics', 'Computers & Accessories', 'C...",Kelby Training DVD: Adobe Photoshop CS5 Crash ...,NaN,NaN,NaN,NaN


In [35]:
#standardization

In [36]:
#Rename First
reviews = reviews.rename(columns={
    'reviewerID': 'user_id',
    'asin': 'item_id',
    'overall': 'rating',
    'unixReviewTime': 'timestamp'
})

In [37]:
reviews = reviews[['user_id', 'item_id', 'rating', 'reviewText', 'summary', 'timestamp']]

In [38]:
meta.columns
meta.head()

,asin,imUrl,description,categories,title,price,salesRank,related,brand
0,0132793040,http://ecx.images-amazon.com/images/I/31JIPhp%...,The Kelby Training DVD Mastering Blend Modes i...,"[['Electronics', 'Computers & Accessories', 'C...",Kelby Training DVD: Mastering Blend Modes in A...,NaN,NaN,NaN,NaN
1,0321732944,http://ecx.images-amazon.com/images/I/31uogm6Y...,NaN,"[['Electronics', 'Computers & Accessories', 'C...",Kelby Training DVD: Adobe Photoshop CS5 Crash ...,NaN,NaN,NaN,NaN
2,0439886341,http://ecx.images-amazon.com/images/I/51k0qa8f...,Digital Organizer and Messenger,"[['Electronics', 'Computers & Accessories', 'P...",Digital Organizer and Messenger,8.15,{'Electronics': 144944},"{'also_viewed': ['0545016266', 'B009ECM8QY', '...",NaN
3,0511189877,http://ecx.images-amazon.com/images/I/41HaAhbv...,The CLIKR-5 UR5U-8780L remote control is desig...,"[['Electronics', 'Accessories & Supplies', 'Au...",CLIKR-5 Time Warner Cable Remote Control UR5U-...,23.36,NaN,"{'also_viewed': ['B001KC08A4', 'B00KUL8O0W', '...",NaN
4,0528881469,http://ecx.images-amazon.com/images/I/51FnRkJq...,"Like its award-winning predecessor, the Intell...","[['Electronics', 'GPS & Navigation', 'Vehicle ...",Rand McNally 528881469 7-inch Intelliroute TND...,299.99,NaN,"{'also_viewed': ['B006ZOI9OY', 'B00C7FKT2A', '...",NaN


In [39]:
# Normailizing Metadat
#description → list / NaN
#categories → nested list
#related → dict / NaN

#into ->clean text + usable structured signals

In [40]:
def clean_description(x):
    if isinstance(x, list):
        return " ".join(x)
    elif isinstance(x, str):
        return x
    else:
        return ""

meta['description'] = meta['description'].apply(clean_description)

In [41]:
#Before: ["good camera", "high quality"]
#After:  "good camera high quality"

In [42]:
def clean_categories(x):
    if isinstance(x, list) and len(x) > 0:
        return " ".join(x[0])   # take first list and flatten
    else:
        return ""

meta['categories'] = meta['categories'].apply(clean_categories)

In [43]:
meta['brand'] = meta['brand'].fillna("")

In [44]:
def extract_related(x):
    if isinstance(x, dict):
        also_bought = x.get('also_bought', [])
        also_viewed = x.get('also_viewed', [])
    else:
        also_bought = []
        also_viewed = []
    return pd.Series([also_bought, also_viewed])

meta[['also_bought', 'also_viewed']] = meta['related'].apply(extract_related)

In [45]:
meta['item_text'] = (
    meta['title'].fillna('') + " " +
    meta['description'] + " " +
    meta['categories'] + " " +
    meta['brand']
)

In [46]:
meta['item_text'] = meta['item_text'].str.lower()

In [49]:
meta = meta.rename(columns={'asin': 'item_id'})

In [50]:
meta[['item_id', 'item_text', 'also_bought', 'also_viewed']].head()

,item_id,item_text,also_bought,also_viewed
0,0132793040,kelby training dvd: mastering blend modes in a...,[],[]
1,0321732944,kelby training dvd: adobe photoshop cs5 crash ...,[],[]
2,0439886341,digital organizer and messenger digital organi...,[],[]
3,0511189877,clikr-5 time warner cable remote control ur5u-...,[],[]
4,0528881469,rand mcnally 528881469 7-inch intelliroute tnd...,[],[]


In [ ]:
# iteam_text for tfidf
# also_bought and viewd for graph signals

In [ ]:
#Merge (create the unified dataset)

In [51]:
merged_df = pd.merge(
    reviews,
    meta,
    on='item_id',
    how='left'
)

In [52]:
merged_df.shape
merged_df.head()

,user_id,item_id,rating,reviewText,summary,timestamp,imUrl,description,categories,title,price,salesRank,related,brand,also_bought,also_viewed,item_text
0,AO94DHGC771SJ,0528881469,5,We got this GPS for my husband who is an (OTR)...,Gotta have GPS!,1370131200,http://ecx.images-amazon.com/images/I/51FnRkJq...,"Like its award-winning predecessor, the Intell...",,Rand McNally 528881469 7-inch Intelliroute TND...,299.99,NaN,"{'also_viewed': ['B006ZOI9OY', 'B00C7FKT2A', '...",,[],[],rand mcnally 528881469 7-inch intelliroute tnd...
1,AMO214LNFCEI4,0528881469,1,"I'm a professional OTR truck driver, and I bou...",Very Disappointed,1290643200,http://ecx.images-amazon.com/images/I/51FnRkJq...,"Like its award-winning predecessor, the Intell...",,Rand McNally 528881469 7-inch Intelliroute TND...,299.99,NaN,"{'also_viewed': ['B006ZOI9OY', 'B00C7FKT2A', '...",,[],[],rand mcnally 528881469 7-inch intelliroute tnd...
2,A3N7T0DY83Y4IG,0528881469,3,"Well, what can I say. I've had this unit in m...",1st impression,1283990400,http://ecx.images-amazon.com/images/I/51FnRkJq...,"Like its award-winning predecessor, the Intell...",,Rand McNally 528881469 7-inch Intelliroute TND...,299.99,NaN,"{'also_viewed': ['B006ZOI9OY', 'B00C7FKT2A', '...",,[],[],rand mcnally 528881469 7-inch intelliroute tnd...
3,A1H8PY3QHMQQA0,0528881469,2,"Not going to write a long review, even thought...","Great grafics, POOR GPS",1290556800,http://ecx.images-amazon.com/images/I/51FnRkJq...,"Like its award-winning predecessor, the Intell...",,Rand McNally 528881469 7-inch Intelliroute TND...,299.99,NaN,"{'also_viewed': ['B006ZOI9OY', 'B00C7FKT2A', '...",,[],[],rand mcnally 528881469 7-inch intelliroute tnd...
4,A24EV6RXELQZ63,0528881469,1,I've had mine for a year and here's what we go...,"Major issues, only excuses for support",1317254400,http://ecx.images-amazon.com/images/I/51FnRkJq...,"Like its award-winning predecessor, the Intell...",,Rand McNally 528881469 7-inch Intelliroute TND...,299.99,NaN,"{'also_viewed': ['B006ZOI9OY', 'B00C7FKT2A', '...",,[],[],rand mcnally 528881469 7-inch intelliroute tnd...


In [53]:
# Check Missing Metadata
merged_df[['item_text']].isnull().sum()

,0
item_text,0


In [54]:
merged_df = merged_df[
    ['user_id', 'item_id', 'rating', 'timestamp',
     'reviewText', 'summary', 'item_text', 'also_bought', 'also_viewed']
]

In [55]:
merged_df.head()
merged_df.columns
merged_df.isnull().sum()

,0
user_id,0
item_id,0
rating,0
timestamp,0
reviewText,0
summary,0
item_text,0
also_bought,0
also_viewed,0


In [56]:
merged_df['item_text'].iloc[0]

'rand mcnally 528881469 7-inch intelliroute tnd 700 truck gps like its award-winning predecessor, the intelliroute tnd 500, the rand mcnally intelliroute tnd 700 gps device gives truckers much more than just maps and navigation. it offers detailed information about truck stops, rest areas, weigh stations and other points of interest for truckers, as well as tools to help drivers estimate the profitability of a route. and with its seven-inch screen and high-definition display, the intelliroute tnd 700 redefines readability and usability. other exclusive features include an oversized stylus, larger buttons, a louder speaker, and tools like a calendar and notepad to help professional drivers complete their routes with greater ease and efficiency.intelliroute tnd 700at a glance:35-percent more truck-routing information than other gps devicesseven-inch, high-definition screenlarge buttons and powerful speaker optimized for truck cabspoken turn-by-turn directionsquick planner tool helps dete

In [ ]:
#Prune & Filter

In [57]:
merged_df = merged_df.drop_duplicates(subset=['user_id', 'item_id'])

In [58]:
#Filter Active Users
user_counts = merged_df['user_id'].value_counts()
active_users = user_counts[user_counts >= 5].index

merged_df = merged_df[merged_df['user_id'].isin(active_users)]

In [59]:
#Filter Active Items
item_counts = merged_df['item_id'].value_counts()
active_items = item_counts[item_counts >= 5].index

merged_df = merged_df[merged_df['item_id'].isin(active_items)]

In [60]:
# Repeat filtering once more
# Re-filter users after item filtering
user_counts = merged_df['user_id'].value_counts()
active_users = user_counts[user_counts >= 5].index

merged_df = merged_df[merged_df['user_id'].isin(active_users)]

In [61]:
merged_df.shape
merged_df['user_id'].nunique()
merged_df['item_id'].nunique()

63001

In [62]:
# Interaction Pipeline for behaviour
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder

# --- Step A1: Extract interaction columns only ---
interactions = merged_df[['user_id', 'item_id', 'rating']].copy()

# --- Step A2: Encode user/item IDs as integers (required for sparse matrix) ---
user_enc = LabelEncoder()
item_enc = LabelEncoder()

interactions['user_idx'] = user_enc.fit_transform(interactions['user_id'])
interactions['item_idx'] = item_enc.fit_transform(interactions['item_id'])

n_users = interactions['user_idx'].nunique()
n_items = interactions['item_idx'].nunique()
print(f"Users: {n_users}, Items: {n_items}")

# --- Step A3: Build sparse user-item matrix (explicit ratings) ---
interaction_matrix = csr_matrix(
    (interactions['rating'].values,
     (interactions['user_idx'].values, interactions['item_idx'].values)),
    shape=(n_users, n_items)
)

print(f"Matrix shape: {interaction_matrix.shape}")
print(f"Sparsity: {1 - interaction_matrix.nnz / (n_users * n_items):.4%}")

# --- Step A4: Save encoders for later (to map back to real IDs) ---
import numpy as np
np.save('user_enc_classes.npy', user_enc.classes_)
np.save('item_enc_classes.npy', item_enc.classes_)

Users: 192403, Items: 63001
Matrix shape: (192403, 63001)
Sparsity: 99.9861%


In [63]:
# Content Pipeline (item features)— Content / TF-IDF matrix
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse

# --- Step B1: item_text is already built (title + desc + categories + brand) ---
# Verify it exists and grab one per item
item_features = merged_df[['item_id', 'item_text']].drop_duplicates('item_id').reset_index(drop=True)
print(f"Unique items with text: {len(item_features)}")

# --- Step B2: Fit TF-IDF ---
tfidf = TfidfVectorizer(
    max_features=10_000,   # keep top 10k terms
    ngram_range=(1, 2),    # unigrams + bigrams
    min_df=2,              # ignore terms in < 2 docs
    sublinear_tf=True      # log-scale TF for stability
)

tfidf_matrix = tfidf.fit_transform(item_features['item_text'])
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")  # (n_items, 10000)

# --- Step B3: Save the item ordering so rows align with item_enc ---
# Re-index rows to match item_enc order
item_id_to_tfidf_row = dict(zip(item_features['item_id'], range(len(item_features))))

# Reorder tfidf_matrix rows to align with item_enc.classes_
ordered_rows = [item_id_to_tfidf_row.get(iid, None) for iid in item_enc.classes_]
valid_mask = [i for i in ordered_rows if i is not None]
tfidf_matrix_ordered = tfidf_matrix[valid_mask]

print(f"Ordered TF-IDF matrix shape: {tfidf_matrix_ordered.shape}")

# --- Step B4: Quick similarity check (cosine) ---
from sklearn.metrics.pairwise import cosine_similarity

sample_item_idx = 0
sim_scores = cosine_similarity(tfidf_matrix_ordered[sample_item_idx], tfidf_matrix_ordered).flatten()
top5 = sim_scores.argsort()[-6:-1][::-1]  # top 5 similar (skip self)
print("Top 5 content-similar items to item 0:")
for idx in top5:
    print(f"  {item_enc.classes_[idx]}: score={sim_scores[idx]:.3f}")

Unique items with text: 63001
TF-IDF matrix shape: (63001, 10000)
Ordered TF-IDF matrix shape: (63001, 10000)
Top 5 content-similar items to item 0:
  B0001MHL0Y: score=0.306
  B003B3P29W: score=0.304
  B003B3P2BU: score=0.300
  B003B3P2CE: score=0.299
  B003BEDTBE: score=0.298


In [64]:
# Pipeline  Popularity / Graph Pipeline (fallbacks & extras)
# C1 simplified — no timestamp needed
item_popularity = (
    interactions.groupby('item_id')
    .agg(
        interaction_count=('rating', 'count'),
        avg_rating=('rating', 'mean')
    )
    .reset_index()
    .sort_values('interaction_count', ascending=False)
)

print("Top 10 popular items:")
print(item_popularity.head(10))

Top 10 popular items:
          item_id  interaction_count  avg_rating
49156  B007WTAJTO               4915    4.587589
29247  B003ES5ZUU               4143    4.800386
59864  B00DR0PDNE               3798    3.997894
16776  B0019EHU8G               3435    4.801164
25965  B002WE6D44               2813    4.659794
29206  B003ELYQGG               2652    4.355958
4763   B0002L5R78               2599    4.599846
54526  B009SYZ8OC               2542    4.444925
57249  B00BGGDVOO               2104    4.421578
25796  B002V88HFE               2082    4.736311


In [65]:
print(merged_df['also_bought'].value_counts().head(10))
print(merged_df['also_bought'].isna().sum())

also_bought
[]    1689188
Name: count, dtype: int64
0


In [66]:
print("=== Pipeline Summary ===")
print(f"A) Interaction matrix : {interaction_matrix.shape} | nnz={interaction_matrix.nnz}")
print(f"B) TF-IDF matrix      : {tfidf_matrix_ordered.shape}")
print(f"C) Popularity table   : {item_popularity.shape}")
print(f"C) Graph edges        : {len(graph_df)}")


=== Pipeline Summary ===
A) Interaction matrix : (192403, 63001) | nnz=1689188
B) TF-IDF matrix      : (63001, 10000)
C) Popularity table   : (63001, 3)


NameError: name 'graph_df' is not defined

In [ ]:
print(merged_df['also_bought'].value_counts().head(10))
print(merged_df['also_bought'].isna().sum())

In [67]:
# Create empty placeholders — graph pipeline skipped gracefully
graph_df = pd.DataFrame(columns=['source', 'target'])
also_bought_map = {}

print("Graph edges: 0 (also_bought empty in this dataset — skipping graph pipeline)")

Graph edges: 0 (also_bought empty in this dataset — skipping graph pipeline)


In [68]:
print("=== Pipeline Summary ===")
print(f"A) Interaction matrix : {interaction_matrix.shape} | nnz={interaction_matrix.nnz}")
print(f"B) TF-IDF matrix      : {tfidf_matrix_ordered.shape}")
print(f"C) Popularity table   : {item_popularity.shape}")
print(f"C) Graph edges        : {len(graph_df)}")

=== Pipeline Summary ===
A) Interaction matrix : (192403, 63001) | nnz=1689188
B) TF-IDF matrix      : (63001, 10000)
C) Popularity table   : (63001, 3)
C) Graph edges        : 0


In [ ]:
!pip install git+https://github.com/lyst/lightfm.git

In [69]:
!pip install implicit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 896.8 kB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for implicit: filename=implicit-0.7.2-cp312-cp312-linux_x86_64.whl size=933264 sha256=4dbb4c4f161f63a646277a1284ad92411c1f43cbd7fd2dd898c777916ee4a3ad
  Stored in directory: /root/.cache/pip/wheels/b2/00/4f/9ff8af07a0a53ac6007ea5d739da19cfe147a2df542b6899f8
Successfully built implicit


In [70]:
#Train/Test Split
import numpy as np
from scipy.sparse import csr_matrix

# Split per user — 80% train, 20% test
np.random.seed(42)

interaction_matrix_lil = interaction_matrix.tolil()  # easier to manipulate
train_matrix = interaction_matrix.copy().tolil()
test_matrix = csr_matrix(interaction_matrix.shape)
test_lil = test_matrix.tolil()

for user_idx in range(interaction_matrix.shape[0]):
    items = interaction_matrix_lil.rows[user_idx]
    if len(items) < 2:
        continue  # skip users with only 1 interaction

    n_test = max(1, int(len(items) * 0.2))
    test_items = np.random.choice(items, n_test, replace=False)

    for item in test_items:
        test_lil[user_idx, item] = interaction_matrix_lil[user_idx, item]
        train_matrix[user_idx, item] = 0  # remove from train

train_csr = csr_matrix(train_matrix)
test_csr = csr_matrix(test_lil)
train_csr.eliminate_zeros()
test_csr.eliminate_zeros()

print(f"Train interactions : {train_csr.nnz}")
print(f"Test interactions  : {test_csr.nnz}")

Train interactions : 1401619
Test interactions  : 287569


In [71]:
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'

In [74]:
import implicit

In [72]:
# How many test items per user on average?
test_items_per_user = np.diff(test_csr.indptr)
print(f"Avg test items per user  : {test_items_per_user[test_items_per_user > 0].mean():.2f}")
print(f"Users with 1 test item   : {(test_items_per_user == 1).sum()}")
print(f"Users with 2+ test items : {(test_items_per_user >= 2).sum()}")
print(f"Users with 5+ test items : {(test_items_per_user >= 5).sum()}")

Avg test items per user  : 1.49
Users with 1 test item   : 147178
Users with 2+ test items : 45225
Users with 5+ test items : 5547


In [75]:
# Retrain — pass user x item matrix directly
# implicit 0.7.2 handles the transpose internally

model = implicit.als.AlternatingLeastSquares(
    factors=64,
    regularization=0.01,
    iterations=20,
    calculate_training_loss=True,
    random_state=42
)

# Pass user x item directly — DO NOT transpose
model.fit(train_csr)

print(f"item_factors shape : {model.item_factors.shape}")   # must be (63001, 64)
print(f"user_factors shape : {model.user_factors.shape}")   # must be (192403, 64)

/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 2 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

item_factors shape : (63001, 64)
user_factors shape : (192403, 64)


In [76]:
!pip install torch --quiet

In [77]:
# Add NCF Model Definition
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class NCF(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=64, hidden=[128, 64]):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)

        layers = []
        input_dim = emb_dim * 2
        for h in hidden:
            layers += [nn.Linear(input_dim, h), nn.ReLU(), nn.Dropout(0.2)]
            input_dim = h
        layers.append(nn.Linear(input_dim, 1))
        self.mlp = nn.Sequential(*layers)

        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)

    def forward(self, user_ids, item_ids):
        u = self.user_emb(user_ids)
        v = self.item_emb(item_ids)
        x = torch.cat([u, v], dim=1)
        return self.mlp(x).squeeze()

print("NCF model defined")

NCF model defined


In [78]:
# Rebuild interactions from merged_df
interactions = merged_df[['user_id', 'item_id', 'rating']].copy()

# Re-encode indices
interactions['user_idx'] = user_enc.transform(interactions['user_id'])
interactions['item_idx'] = item_enc.transform(interactions['item_id'])

print(f"Interactions shape : {interactions.shape}")
print(f"Columns            : {interactions.columns.tolist()}")
print(f"Sample:")
print(interactions.head(3))

Interactions shape : (1689188, 5)
Columns            : ['user_id', 'item_id', 'rating', 'user_idx', 'item_idx']
Sample:
          user_id     item_id  rating  user_idx  item_idx
0   AO94DHGC771SJ  0528881469       5    176008         0
1   AMO214LNFCEI4  0528881469       1    173739         0
2  A3N7T0DY83Y4IG  0528881469       3    134504         0


In [79]:
# Build Dataset
from sklearn.model_selection import train_test_split

class InteractionDataset(Dataset):
    def __init__(self, df):
        self.users   = torch.tensor(df['user_idx'].values,  dtype=torch.long)
        self.items   = torch.tensor(df['item_idx'].values,  dtype=torch.long)
        self.ratings = torch.tensor(
            (df['rating'].values - 1) / 4, dtype=torch.float32  # normalize 1-5 → 0-1
        )
    def __len__(self): return len(self.users)
    def __getitem__(self, idx): return self.users[idx], self.items[idx], self.ratings[idx]

# Split — reuses your existing interactions dataframe
train_df, test_df = train_test_split(interactions, test_size=0.2, random_state=42)

train_loader = DataLoader(InteractionDataset(train_df), batch_size=1024, shuffle=True)
test_loader  = DataLoader(InteractionDataset(test_df),  batch_size=1024, shuffle=False)

print(f"Train batches : {len(train_loader)}")
print(f"Test batches  : {len(test_loader)}")

Train batches : 1320
Test batches  : 330


In [81]:
#Train NCF
n_users = interactions['user_idx'].nunique()
n_items = interactions['item_idx'].nunique()

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ncf_model = NCF(n_users=n_users, n_items=n_items).to(device)
optimizer = torch.optim.Adam(ncf_model.parameters(), lr=0.001)
criterion = nn.MSELoss()

print(f"Training on : {device}")
print(f"Users: {n_users}, Items: {n_items}")

def train_epoch(model, loader):
    model.train()
    total = 0
    for users, items, ratings in loader:
        users, items, ratings = users.to(device), items.to(device), ratings.to(device)
        optimizer.zero_grad()
        loss = criterion(model(users, items), ratings)
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total / len(loader)

def eval_epoch(model, loader):
    model.eval()
    total = 0
    with torch.no_grad():
        for users, items, ratings in loader:
            users, items, ratings = users.to(device), items.to(device), ratings.to(device)
            total += criterion(model(users, items), ratings).item()
    return total / len(loader)

# Train 10 epochs
for epoch in range(10):
    tr = train_epoch(ncf_model, train_loader)
    vl = eval_epoch(ncf_model, test_loader)
    print(f"Epoch {epoch+1:2d} | Train Loss: {tr:.4f} | Val Loss: {vl:.4f}")

Training on : cpu
Users: 192403, Items: 63001
Epoch  1 | Train Loss: 0.0893 | Val Loss: 0.0754
Epoch  2 | Train Loss: 0.0675 | Val Loss: 0.0757
Epoch  3 | Train Loss: 0.0580 | Val Loss: 0.0783
Epoch  4 | Train Loss: 0.0497 | Val Loss: 0.0828
Epoch  5 | Train Loss: 0.0402 | Val Loss: 0.0880
Epoch  6 | Train Loss: 0.0309 | Val Loss: 0.0918
Epoch  7 | Train Loss: 0.0242 | Val Loss: 0.0968
Epoch  8 | Train Loss: 0.0199 | Val Loss: 0.0996
Epoch  9 | Train Loss: 0.0170 | Val Loss: 0.0987
Epoch 10 | Train Loss: 0.0150 | Val Loss: 0.1006


In [ ]:
#Overfitting — val loss goes up while train loss goes down:
#Epoch 1: Train 0.0893 Val 0.0754  ← good
#Epoch 5: Train 0.0402 Val 0.0880  ← diverging
#Epoch 10: Train 0.0150 Val 0.1006 ← overfit

In [83]:
# Restore score function
def score(user_id, item_id):
    try:
        user_idx = np.where(user_enc.classes_ == user_id)[0][0]
        item_idx = np.where(item_enc.classes_ == item_id)[0][0]
    except IndexError:
        return 0.0
    u = model.user_factors[user_idx]
    v = model.item_factors[item_idx]
    return float(np.dot(u, v))

# Restore sample variables
active_users = np.where(np.diff(train_csr.indptr) > 0)[0]
sample_user  = user_enc.classes_[active_users[0]]
sample_item  = item_enc.classes_[0]

print(f"sample_user : {sample_user}")
print(f"sample_item : {sample_item}")
print(f"ALS score   : {score(sample_user, sample_item):.4f}")

sample_user : A000715434M800HLCENK9
sample_item : 0528881469
ALS score   : -0.0003


In [84]:

# NCF Score function (replaces ALS score)
def ncf_score(user_id, item_id):
    """
    Drop-in replacement for your existing score() function.
    Same inputs, same output — just uses NCF instead of ALS.
    """
    try:
        user_idx = np.where(user_enc.classes_ == user_id)[0][0]
        item_idx = np.where(item_enc.classes_ == item_id)[0][0]
    except IndexError:
        return 0.0

    ncf_model.eval()
    with torch.no_grad():
        u = torch.tensor([user_idx], dtype=torch.long).to(device)
        v = torch.tensor([item_idx], dtype=torch.long).to(device)
        return ncf_model(u, v).item()

# Quick test
print(f"ALS score : {score(sample_user, sample_item):.4f}")
print(f"NCF score : {ncf_score(sample_user, sample_item):.4f}")

ALS score : -0.0003
NCF score : 0.3831


In [85]:
#Compare ALS vs NCF
def evaluate_ncf(n_users=1000, k=10):
    test_users  = np.where(np.diff(test_csr.indptr) > 0)[0]
    train_users = np.where(np.diff(train_csr.indptr) > 0)[0]
    valid_users = np.intersect1d(test_users, train_users)
    sample      = np.random.choice(valid_users, min(n_users, len(valid_users)), replace=False)

    als_p, ncf_p = [], []
    als_r, ncf_r = [], []

    ncf_model.eval()
    for user_idx in sample:
        actual = set(test_csr[user_idx].indices.tolist())
        if not actual:
            continue
        try:
            # ALS recommendations
            ids, _ = model.recommend(
                user_idx, train_csr[user_idx],
                N=k, filter_already_liked_items=True
            )
            hits = len(actual & set(ids.tolist()))
            als_p.append(hits / k)
            als_r.append(hits / len(actual))

            # NCF recommendations — score all items, pick top-K
            user_id = user_enc.classes_[user_idx]
            train_items = set(train_csr[user_idx].indices.tolist())

            # Score 500 random unseen items (full scoring too slow)
            all_items  = list(set(range(n_items)) - train_items)
            sample_items = np.random.choice(all_items, min(500, len(all_items)), replace=False)

            with torch.no_grad():
                u = torch.tensor([user_idx]*len(sample_items), dtype=torch.long).to(device)
                v = torch.tensor(sample_items, dtype=torch.long).to(device)
                scores_ncf = ncf_model(u, v).cpu().numpy()

            top_k_idx = sample_items[scores_ncf.argsort()[::-1][:k]]
            hits_ncf  = len(actual & set(top_k_idx.tolist()))
            ncf_p.append(hits_ncf / k)
            ncf_r.append(hits_ncf / len(actual))

        except Exception:
            continue

    print("=== ALS vs NCF ===")
    print(f"ALS  Precision@{k}: {np.mean(als_p):.4f} | Recall@{k}: {np.mean(als_r):.4f}")
    print(f"NCF  Precision@{k}: {np.mean(ncf_p):.4f} | Recall@{k}: {np.mean(ncf_r):.4f}")

evaluate_ncf()

=== ALS vs NCF ===
ALS  Precision@10: 0.0046 | Recall@10: 0.0323
NCF  Precision@10: 0.0000 | Recall@10: 0.0000


In [ ]:
#Issue 2 — NCF Precision 0.0000 — same sampling problem as before, scoring only 500 random items misses actual test items.


In [88]:
# Retrain with early stopping + regularization
# Retrain NCF with fixes
ncf_model = NCF(n_users=n_users, n_items=n_items, emb_dim=64, hidden=[128, 64]).to(device)
optimizer  = torch.optim.Adam(ncf_model.parameters(), lr=0.001, weight_decay=1e-5)
criterion  = nn.MSELoss()

best_val_loss = float('inf')
patience      = 3
no_improve    = 0
best_weights  = None

print("Retraining NCF with early stopping...")
for epoch in range(20):
    tr = train_epoch(ncf_model, train_loader)
    vl = eval_epoch(ncf_model, test_loader)
    print(f"Epoch {epoch+1:2d} | Train: {tr:.4f} | Val: {vl:.4f}", end="")

    if vl < best_val_loss:
        best_val_loss = vl
        best_weights  = {k: v.clone() for k, v in ncf_model.state_dict().items()}
        no_improve    = 0
        print(" ✅ best")
    else:
        no_improve += 1
        print(f" (no improve {no_improve}/{patience})")

    if no_improve >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

# Restore best weights
ncf_model.load_state_dict(best_weights)
print(f"\nRestored best model (val loss: {best_val_loss:.4f})")

Retraining NCF with early stopping...
Epoch  1 | Train: 0.0949 | Val: 0.0768 ✅ best
Epoch  2 | Train: 0.0743 | Val: 0.0751 ✅ best
Epoch  3 | Train: 0.0669 | Val: 0.0749 ✅ best
Epoch  4 | Train: 0.0621 | Val: 0.0760 (no improve 1/3)
Epoch  5 | Train: 0.0588 | Val: 0.0779 (no improve 2/3)
Epoch  6 | Train: 0.0563 | Val: 0.0792 (no improve 3/3)

Early stopping at epoch 6

Restored best model (val loss: 0.0749)


In [90]:
#Now plug NCF into your router
#Replace score() with NCF version
# Override the old ALS score function with NCF
def score(user_id, item_id):
    """Now powered by NCF instead of ALS"""
    try:
        user_idx = np.where(user_enc.classes_ == user_id)[0][0]
        item_idx = np.where(item_enc.classes_ == item_id)[0][0]
    except IndexError:
        return 0.0
    ncf_model.eval()
    with torch.no_grad():
        u = torch.tensor([user_idx], dtype=torch.long).to(device)
        v = torch.tensor([item_idx], dtype=torch.long).to(device)
        return ncf_model(u, v).item()

print(f"score() now uses NCF: {score(sample_user, sample_item):.4f}")

score() now uses NCF: 0.4311


In [91]:
#Replace recommend() candidate generation with NCF
def ncf_recommend(user_id, top_n=10):
    """
    Full NCF-powered recommendations.
    Scores all unseen items and returns top-N.
    """
    try:
        user_idx = np.where(user_enc.classes_ == user_id)[0][0]
    except IndexError:
        return []

    # Get items user hasn't seen
    train_items = set(train_csr[user_idx].indices.tolist())
    all_items   = np.array(
        [i for i in range(len(item_enc.classes_)) if i not in train_items]
    )

    # Score all unseen items in batches
    ncf_model.eval()
    all_scores = []
    batch_size = 2048

    with torch.no_grad():
        for i in range(0, len(all_items), batch_size):
            batch = all_items[i:i+batch_size]
            u     = torch.tensor([user_idx]*len(batch), dtype=torch.long).to(device)
            v     = torch.tensor(batch, dtype=torch.long).to(device)
            s     = ncf_model(u, v).cpu().numpy()
            all_scores.extend(s)

    all_scores = np.array(all_scores)
    top_idx    = all_scores.argsort()[::-1][:top_n]
    top_items  = all_items[top_idx]

    return [{
        'item_id': item_enc.classes_[i],
        'title'  : item_features[
            item_features['item_id'] == item_enc.classes_[i]
        ]['item_text'].values[0][:70],
        'score'  : round(float(all_scores[top_idx[rank]]), 4),
        'source' : 'NCF'
    } for rank, i in enumerate(top_items)]


# Test it
print("\nNCF Recommendations:")
print("=" * 55)
recs = ncf_recommend(sample_user, top_n=5)
for i, r in enumerate(recs, 1):
    print(f"  {i}. [{r['score']:.4f}] {r['title']}...")


NCF Recommendations:
  1. [0.8800] canon ef 135mm f/2l usm lens for canon slr cameras the fastest 135mm t...
  2. [0.8759] nikon 70-200mm f/4g ed vr nikkor zoom lens   nikon...
  3. [0.8738] bluerigger digital optical audio toslink cable (25 feet)- cl3 rated th...
  4. [0.8716] manfrotto 035rl super clamp with 2908 standard stud - replaces 2900 - ...
  5. [0.8689] nikon 85mm f/1.4g af-s nikkor lens for nikon digital slr this updated ...


In [93]:
# ── MASTER CELL — run this after any reset ──
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def get_user_interaction_count(user_id):
    user_idx = np.where(user_enc.classes_ == user_id)[0]
    if len(user_idx) == 0:
        return 0
    return int(train_csr[user_idx[0]].nnz)

def get_recommendation_confidence(user_id, recs):
    if not recs:
        return 0.0
    return np.mean([r['score'] for r in recs])

class ElectronicsRecommender:
    def __init__(self, model, user_enc, item_enc, train_csr,
                 tfidf_matrix, item_features, item_popularity, popularity_norm):
        self.model           = model
        self.user_enc        = user_enc
        self.item_enc        = item_enc
        self.train_csr       = train_csr
        self.tfidf_matrix    = tfidf_matrix
        self.item_features   = item_features
        self.item_popularity = item_popularity
        self.popularity_norm = popularity_norm

    def _user_idx(self, user_id):
        idx = np.where(self.user_enc.classes_ == user_id)[0]
        return idx[0] if len(idx) > 0 else None

    def _item_idx(self, item_id):
        idx = np.where(self.item_enc.classes_ == item_id)[0]
        return idx[0] if len(idx) > 0 else None

    def _item_title(self, item_id):
        row = self.item_features[
            self.item_features['item_id'] == item_id
        ]['item_text'].values
        return row[0][:70] if len(row) > 0 else 'unknown'

    def _popularity_fallback(self, top_n):
        results = []
        for _, row in self.item_popularity.head(top_n).iterrows():
            results.append({
                'item_id': row['item_id'],
                'title'  : self._item_title(row['item_id']),
                'score'  : round(self.popularity_norm.get(row['item_id'], 0.0), 4),
                'source' : 'popularity'
            })
        return results

    def _als_score(self, user_idx, item_id):
        item_idx = self._item_idx(item_id)
        if item_idx is None:
            return 0.0
        u = self.model.user_factors[user_idx]
        v = self.model.item_factors[item_idx]
        return float(np.dot(u, v))

    def _content_score(self, user_idx, item_id):
        item_idx = self._item_idx(item_id)
        if item_idx is None:
            return 0.0
        user_items = self.train_csr[user_idx].indices[:20]
        if len(user_items) == 0:
            return 0.0
        target  = self.tfidf_matrix[item_idx]
        history = self.tfidf_matrix[user_items]
        return float(np.mean(cosine_similarity(target, history).flatten()))

    def _hybrid_score(self, user_id, item_id, alpha, beta, gamma):
        user_idx = self._user_idx(user_id)
        if user_idx is None:
            return 0.0
        als     = (self._als_score(user_idx, item_id) + 1) / 2
        content = self._content_score(user_idx, item_id)
        pop     = self.popularity_norm.get(item_id, 0.0)
        return alpha * als + beta * content + gamma * pop

    def recommend(self, user_id, top_n=10, alpha=0.6, beta=0.3, gamma=0.1):
        user_idx = self._user_idx(user_id)
        if user_idx is None:
            return self._popularity_fallback(top_n)
        ids, _ = self.model.recommend(
            user_idx, self.train_csr[user_idx],
            N=200, filter_already_liked_items=True
        )
        candidates = [self.item_enc.classes_[i] for i in ids]
        scored = sorted(
            [(iid, self._hybrid_score(user_id, iid, alpha, beta, gamma))
             for iid in candidates],
            key=lambda x: x[1], reverse=True
        )[:top_n]
        return [{
            'item_id': iid,
            'title'  : self._item_title(iid),
            'score'  : round(s, 4),
            'source' : 'hybrid'
        } for iid, s in scored]

    def similar(self, item_id, top_n=10):
        item_idx = self._item_idx(item_id)
        if item_idx is not None:
            item_vec = self.model.item_factors[item_idx].reshape(1, -1)
            sims     = cosine_similarity(item_vec, self.model.item_factors).flatten()
            top_idx  = sims.argsort()[::-1][1:top_n+1]
            if sims[top_idx[0]] > 0.1:
                return [{
                    'item_id'   : self.item_enc.classes_[i],
                    'title'     : self._item_title(self.item_enc.classes_[i]),
                    'similarity': round(float(sims[i]), 4),
                    'source'    : 'ALS'
                } for i in top_idx]
        row = self.item_features[self.item_features['item_id'] == item_id]
        if row.empty:
            return []
        idx      = row.index[0]
        item_vec = self.tfidf_matrix[idx]
        sims     = cosine_similarity(item_vec, self.tfidf_matrix).flatten()
        top_idx  = sims.argsort()[::-1][1:top_n+1]
        return [{
            'item_id'   : self.item_features.iloc[i]['item_id'],
            'title'     : self.item_features.iloc[i]['item_text'][:70],
            'similarity': round(float(sims[i]), 4),
            'source'    : 'TF-IDF'
        } for i in top_idx]

    def trending(self, top_n=10, category=None):
        if category:
            mask  = self.item_features['item_text'].str.lower().str.contains(
                category.lower(), na=False
            )
            items = self.item_features[mask]['item_id'].tolist()
            pool  = self.item_popularity[
                self.item_popularity['item_id'].isin(items)
            ].head(top_n)
        else:
            pool = self.item_popularity.head(top_n)
        return [{
            'item_id'          : row['item_id'],
            'title'            : self._item_title(row['item_id']),
            'interaction_count': int(row['interaction_count']),
            'avg_rating'       : round(float(row['avg_rating']), 2),
            'score'            : round(self.popularity_norm.get(row['item_id'], 0.0), 4),
            'source'           : 'trending'
        } for _, row in pool.iterrows()]

    def score(self, user_id, item_id):
        return round(self._hybrid_score(user_id, item_id, 0.6, 0.3, 0.1), 4)


class RecommendationRouter:
    NEW_USER_THRESHOLD   = 0
    COLD_USER_THRESHOLD  = 5
    WARM_USER_THRESHOLD  = 20
    CONFIDENCE_THRESHOLD = 0.35

    def __init__(self, recommender):
        self.rec = recommender

    def _classify_user(self, user_id):
        n = get_user_interaction_count(user_id)
        if n == 0:       return 'new',    n
        elif n < self.COLD_USER_THRESHOLD: return 'cold', n
        elif n < self.WARM_USER_THRESHOLD: return 'warm', n
        else:            return 'active', n

    def _blend_with_popularity(self, personalized, top_n, blend_ratio=0.4):
        n_popular      = max(1, int(top_n * blend_ratio))
        n_personalized = top_n - n_popular
        popular        = self.rec.trending(top_n=n_popular)
        personalized_ids = {r['item_id'] for r in personalized}
        popular_filtered = [
            r for r in popular if r['item_id'] not in personalized_ids
        ][:n_popular]
        for r in personalized[:n_personalized]: r['strategy'] = 'hybrid'
        for r in popular_filtered:
            r['strategy'] = 'popularity_blend'
            r['score']    = r.get('score', 0.0)
        return personalized[:n_personalized] + popular_filtered

    def for_user(self, user_id, top_n=10):
        user_tier, n_interactions = self._classify_user(user_id)
        print(f"[ROUTER] User tier : {user_tier} ({n_interactions} interactions)")
        if user_tier == 'new':
            print(f"[ROUTER] Strategy  : Trending")
            results = self.rec.trending(top_n=top_n)
            for r in results: r['strategy'] = 'trending'
            return results
        if user_tier == 'cold':
            print(f"[ROUTER] Strategy  : Hybrid + 60% popularity blend")
            return self._blend_with_popularity(
                self.rec.recommend(user_id, top_n=top_n), top_n, blend_ratio=0.6
            )
        personalized = self.rec.recommend(user_id, top_n=top_n)
        confidence   = get_recommendation_confidence(user_id, personalized)
        print(f"[ROUTER] Strategy  : {'Warm' if user_tier=='warm' else 'Full'} hybrid (confidence={confidence:.4f})")
        if confidence < self.CONFIDENCE_THRESHOLD:
            return self._blend_with_popularity(
                personalized, top_n,
                blend_ratio=0.3 if user_tier == 'warm' else 0.2
            )
        return personalized

    def for_product_page(self, item_id, top_n=10):
        print(f"[ROUTER] Product page : {item_id}")
        similar = self.rec.similar(item_id, top_n=top_n)
        if not similar:
            results = self.rec.trending(top_n=top_n)
            for r in results: r['strategy'] = 'popularity_fallback'
            return results
        for r in similar: r['strategy'] = similar[0].get('source', 'unknown')
        return similar

    def explain(self, user_id):
        user_tier, n = self._classify_user(user_id)
        print(f"\n=== Routing for {user_id} ===")
        print(f"  Interactions : {n}")
        print(f"  Tier         : {user_tier}")
        strategies = {
            'new'   : 'Trending',
            'cold'  : 'Hybrid + 60% popularity blend',
            'warm'  : 'Hybrid + confidence check',
            'active': 'Full hybrid'
        }
        print(f"  Strategy     : {strategies[user_tier]}")


# ── Instantiate ──
active_users = np.where(np.diff(train_csr.indptr) > 0)[0]
sample_user  = user_enc.classes_[active_users[0]]
sample_item  = item_enc.classes_[0]

rec    = ElectronicsRecommender(
    model=model, user_enc=user_enc, item_enc=item_enc,
    train_csr=train_csr, tfidf_matrix=tfidf_matrix_ordered,
    item_features=item_features, item_popularity=item_popularity,
    popularity_norm=popularity_norm
)
router = RecommendationRouter(rec)
print("✅ All classes and instances ready")

NameError: name 'popularity_norm' is not defined

In [95]:
# ── MASTER CELL — run this after any reset ──
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def get_user_interaction_count(user_id):
    user_idx = np.where(user_enc.classes_ == user_id)[0]
    if len(user_idx) == 0:
        return 0
    return int(train_csr[user_idx[0]].nnz)

def get_recommendation_confidence(user_id, recs):
    if not recs:
        return 0.0
    return np.mean([r['score'] for r in recs])

class ElectronicsRecommender:
    def __init__(self, model, user_enc, item_enc, train_csr,
                 tfidf_matrix, item_features, item_popularity, popularity_norm):
        self.model           = model
        self.user_enc        = user_enc
        self.item_enc        = item_enc
        self.train_csr       = train_csr
        self.tfidf_matrix    = tfidf_matrix
        self.item_features   = item_features
        self.item_popularity = item_popularity
        self.popularity_norm = popularity_norm

    def _user_idx(self, user_id):
        idx = np.where(self.user_enc.classes_ == user_id)[0]
        return idx[0] if len(idx) > 0 else None

    def _item_idx(self, item_id):
        idx = np.where(self.item_enc.classes_ == item_id)[0]
        return idx[0] if len(idx) > 0 else None

    def _item_title(self, item_id):
        row = self.item_features[
            self.item_features['item_id'] == item_id
        ]['item_text'].values
        return row[0][:70] if len(row) > 0 else 'unknown'

    def _popularity_fallback(self, top_n):
        results = []
        for _, row in self.item_popularity.head(top_n).iterrows():
            results.append({
                'item_id': row['item_id'],
                'title'  : self._item_title(row['item_id']),
                'score'  : round(self.popularity_norm.get(row['item_id'], 0.0), 4),
                'source' : 'popularity'
            })
        return results

    def _als_score(self, user_idx, item_id):
        item_idx = self._item_idx(item_id)
        if item_idx is None:
            return 0.0
        u = self.model.user_factors[user_idx]
        v = self.model.item_factors[item_idx]
        return float(np.dot(u, v))

    def _content_score(self, user_idx, item_id):
        item_idx = self._item_idx(item_id)
        if item_idx is None:
            return 0.0
        user_items = self.train_csr[user_idx].indices[:20]
        if len(user_items) == 0:
            return 0.0
        target  = self.tfidf_matrix[item_idx]
        history = self.tfidf_matrix[user_items]
        return float(np.mean(cosine_similarity(target, history).flatten()))

    def _hybrid_score(self, user_id, item_id, alpha, beta, gamma):
        user_idx = self._user_idx(user_id)
        if user_idx is None:
            return 0.0
        als     = (self._als_score(user_idx, item_id) + 1) / 2
        content = self._content_score(user_idx, item_id)
        pop     = self.popularity_norm.get(item_id, 0.0)
        return alpha * als + beta * content + gamma * pop

    def recommend(self, user_id, top_n=10, alpha=0.6, beta=0.3, gamma=0.1):
        user_idx = self._user_idx(user_id)
        if user_idx is None:
            return self._popularity_fallback(top_n)
        ids, _ = self.model.recommend(
            user_idx, self.train_csr[user_idx],
            N=200, filter_already_liked_items=True
        )
        candidates = [self.item_enc.classes_[i] for i in ids]
        scored = sorted(
            [(iid, self._hybrid_score(user_id, iid, alpha, beta, gamma))
             for iid in candidates],
            key=lambda x: x[1], reverse=True
        )[:top_n]
        return [{
            'item_id': iid,
            'title'  : self._item_title(iid),
            'score'  : round(s, 4),
            'source' : 'hybrid'
        } for iid, s in scored]

    def similar(self, item_id, top_n=10):
        item_idx = self._item_idx(item_id)
        if item_idx is not None:
            item_vec = self.model.item_factors[item_idx].reshape(1, -1)
            sims     = cosine_similarity(item_vec, self.model.item_factors).flatten()
            top_idx  = sims.argsort()[::-1][1:top_n+1]
            if sims[top_idx[0]] > 0.1:
                return [{
                    'item_id'   : self.item_enc.classes_[i],
                    'title'     : self._item_title(self.item_enc.classes_[i]),
                    'similarity': round(float(sims[i]), 4),
                    'source'    : 'ALS'
                } for i in top_idx]
        row = self.item_features[self.item_features['item_id'] == item_id]
        if row.empty:
            return []
        idx      = row.index[0]
        item_vec = self.tfidf_matrix[idx]
        sims     = cosine_similarity(item_vec, self.tfidf_matrix).flatten()
        top_idx  = sims.argsort()[::-1][1:top_n+1]
        return [{
            'item_id'   : self.item_features.iloc[i]['item_id'],
            'title'     : self.item_features.iloc[i]['item_text'][:70],
            'similarity': round(float(sims[i]), 4),
            'source'    : 'TF-IDF'
        } for i in top_idx]

    def trending(self, top_n=10, category=None):
        if category:
            mask  = self.item_features['item_text'].str.lower().str.contains(
                category.lower(), na=False
            )
            items = self.item_features[mask]['item_id'].tolist()
            pool  = self.item_popularity[
                self.item_popularity['item_id'].isin(items)
            ].head(top_n)
        else:
            pool = self.item_popularity.head(top_n)
        return [{
            'item_id'          : row['item_id'],
            'title'            : self._item_title(row['item_id']),
            'interaction_count': int(row['interaction_count']),
            'avg_rating'       : round(float(row['avg_rating']), 2),
            'score'            : round(self.popularity_norm.get(row['item_id'], 0.0), 4),
            'source'           : 'trending'
        } for _, row in pool.iterrows()]

    def score(self, user_id, item_id):
        return round(self._hybrid_score(user_id, item_id, 0.6, 0.3, 0.1), 4)


class RecommendationRouter:
    NEW_USER_THRESHOLD   = 0
    COLD_USER_THRESHOLD  = 5
    WARM_USER_THRESHOLD  = 20
    CONFIDENCE_THRESHOLD = 0.35

    def __init__(self, recommender):
        self.rec = recommender

    def _classify_user(self, user_id):
        n = get_user_interaction_count(user_id)
        if n == 0:       return 'new',    n
        elif n < self.COLD_USER_THRESHOLD: return 'cold', n
        elif n < self.WARM_USER_THRESHOLD: return 'warm', n
        else:            return 'active', n

    def _blend_with_popularity(self, personalized, top_n, blend_ratio=0.4):
        n_popular      = max(1, int(top_n * blend_ratio))
        n_personalized = top_n - n_popular
        popular        = self.rec.trending(top_n=n_popular)
        personalized_ids = {r['item_id'] for r in personalized}
        popular_filtered = [
            r for r in popular if r['item_id'] not in personalized_ids
        ][:n_popular]
        for r in personalized[:n_personalized]: r['strategy'] = 'hybrid'
        for r in popular_filtered:
            r['strategy'] = 'popularity_blend'
            r['score']    = r.get('score', 0.0)
        return personalized[:n_personalized] + popular_filtered

    def for_user(self, user_id, top_n=10):
        user_tier, n_interactions = self._classify_user(user_id)
        print(f"[ROUTER] User tier : {user_tier} ({n_interactions} interactions)")
        if user_tier == 'new':
            print(f"[ROUTER] Strategy  : Trending")
            results = self.rec.trending(top_n=top_n)
            for r in results: r['strategy'] = 'trending'
            return results
        if user_tier == 'cold':
            print(f"[ROUTER] Strategy  : Hybrid + 60% popularity blend")
            return self._blend_with_popularity(
                self.rec.recommend(user_id, top_n=top_n), top_n, blend_ratio=0.6
            )
        personalized = self.rec.recommend(user_id, top_n=top_n)
        confidence   = get_recommendation_confidence(user_id, personalized)
        print(f"[ROUTER] Strategy  : {'Warm' if user_tier=='warm' else 'Full'} hybrid (confidence={confidence:.4f})")
        if confidence < self.CONFIDENCE_THRESHOLD:
            return self._blend_with_popularity(
                personalized, top_n,
                blend_ratio=0.3 if user_tier == 'warm' else 0.2
            )
        return personalized

    def for_product_page(self, item_id, top_n=10):
        print(f"[ROUTER] Product page : {item_id}")
        similar = self.rec.similar(item_id, top_n=top_n)
        if not similar:
            results = self.rec.trending(top_n=top_n)
            for r in results: r['strategy'] = 'popularity_fallback'
            return results
        for r in similar: r['strategy'] = similar[0].get('source', 'unknown')
        return similar

    def explain(self, user_id):
        user_tier, n = self._classify_user(user_id)
        print(f"\n=== Routing for {user_id} ===")
        print(f"  Interactions : {n}")
        print(f"  Tier         : {user_tier}")
        strategies = {
            'new'   : 'Trending',
            'cold'  : 'Hybrid + 60% popularity blend',
            'warm'  : 'Hybrid + confidence check',
            'active': 'Full hybrid'
        }
        print(f"  Strategy     : {strategies[user_tier]}")


# ── Instantiate ──
active_users = np.where(np.diff(train_csr.indptr) > 0)[0]
sample_user  = user_enc.classes_[active_users[0]]
sample_item  = item_enc.classes_[0]

rec    = ElectronicsRecommender(
    model=model, user_enc=user_enc, item_enc=item_enc,
    train_csr=train_csr, tfidf_matrix=tfidf_matrix_ordered,
    item_features=item_features, item_popularity=item_popularity,
    popularity_norm=popularity_norm
)
router = RecommendationRouter(rec)
print("✅ All classes and instances ready")

✅ All classes and instances ready


In [96]:
# Rebuild all missing variables in one cell
import numpy as np

# Popularity norm
max_pop      = item_popularity['interaction_count'].max()
popularity_norm = {
    row['item_id']: row['interaction_count'] / max_pop
    for _, row in item_popularity.iterrows()
}

# Sample variables
active_users = np.where(np.diff(train_csr.indptr) > 0)[0]
sample_user  = user_enc.classes_[active_users[0]]
sample_item  = item_enc.classes_[0]

print(f"popularity_norm : {len(popularity_norm)} items")
print(f"sample_user     : {sample_user}")
print(f"sample_item     : {sample_item}")
print("✅ All variables restored")

popularity_norm : 63001 items
sample_user     : A000715434M800HLCENK9
sample_item     : 0528881469
✅ All variables restored


In [97]:
#Update ElectronicsRecommender to use NCF
# Patch the recommender class to use NCF
class NCFElectronicsRecommender(ElectronicsRecommender):
    """
    Extends ElectronicsRecommender — replaces ALS with NCF backbone.
    Everything else (similar, trending, routing) stays the same.
    """
    def __init__(self, ncf_model, device, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.ncf_model = ncf_model
        self.device    = device

    def _als_score(self, user_idx, item_id):
        """Override — use NCF instead of ALS dot product"""
        item_idx = self._item_idx(item_id)
        if item_idx is None:
            return 0.0
        self.ncf_model.eval()
        with torch.no_grad():
            u = torch.tensor([user_idx], dtype=torch.long).to(self.device)
            v = torch.tensor([item_idx], dtype=torch.long).to(self.device)
            return self.ncf_model(u, v).item()

    def recommend(self, user_id, top_n=10, alpha=0.6, beta=0.3, gamma=0.1):
        """Override — use NCF for candidate generation"""
        user_idx = self._user_idx(user_id)
        if user_idx is None:
            return self._popularity_fallback(top_n)
        return ncf_recommend(user_id, top_n=top_n)


# Instantiate NCF-powered recommender
ncf_rec = NCFElectronicsRecommender(
    ncf_model       = ncf_model,
    device          = device,
    model           = model,
    user_enc        = user_enc,
    item_enc        = item_enc,
    train_csr       = train_csr,
    tfidf_matrix    = tfidf_matrix_ordered,
    item_features   = item_features,
    item_popularity = item_popularity,
    popularity_norm = popularity_norm
)

# Update router to use NCF recommender
ncf_router = RecommendationRouter(ncf_rec)
print("✅ NCF recommender and router ready")

✅ NCF recommender and router ready


In [98]:
print("=== Final System Test ===")

print("\n── ALS Router ──")
als_recs = router.for_user(sample_user, top_n=5)
for i, r in enumerate(als_recs, 1):
    print(f"  {i}. [{r.get('score',0):.4f}] {r['title'][:50]}...")

print("\n── NCF Router ──")
ncf_recs = ncf_router.for_user(sample_user, top_n=5)
for i, r in enumerate(ncf_recs, 1):
    print(f"  {i}. [{r.get('score',0):.4f}] {r['title'][:50]}...")

=== Final System Test ===

── ALS Router ──
[ROUTER] User tier : cold (4 interactions)
[ROUTER] Strategy  : Hybrid + 60% popularity blend
  1. [0.4168] sandisk ultra 64gb microsdxc class 10 uhs memory c...
  2. [0.3888] amazonbasics digital optical audio toslink cable, ...
  3. [0.8429] amazonbasics high-speed hdmi cable - 15 feet ( 4.6...
  4. [0.7727] google chromecast hdmi streaming media player   go...

── NCF Router ──
[ROUTER] User tier : cold (4 interactions)
[ROUTER] Strategy  : Hybrid + 60% popularity blend
  1. [0.8800] canon ef 135mm f/2l usm lens for canon slr cameras...
  2. [0.8759] nikon 70-200mm f/4g ed vr nikkor zoom lens   nikon...
  3. [1.0000] sandisk ultra 64gb microsdxc class 10 uhs memory c...
  4. [0.8429] amazonbasics high-speed hdmi cable - 15 feet ( 4.6...
  5. [0.7727] google chromecast hdmi streaming media player   go...


In [ ]:
# Service Functions

In [99]:
#recommend()
# Get personalized recommendations for any user
user_id = user_enc.classes_[active_users[5]]  # pick any known user

print(f"Recommendations for: {user_id}")
print("=" * 55)
recs = ncf_router.for_user(user_id, top_n=10)
for i, r in enumerate(recs, 1):
    print(f"  {i:2}. [{r.get('score',0):.4f}] {r['title'][:55]}...")

# Try with a new unknown user
print("\nRecommendations for new user:")
print("=" * 55)
recs = ncf_router.for_user("BRAND_NEW_USER", top_n=5)
for i, r in enumerate(recs, 1):
    print(f"  {i:2}. [{r.get('score',0):.4f}] {r['title'][:55]}...")

Recommendations for: A00473363TJ8YSZ3YAGG9
[ROUTER] User tier : warm (8 interactions)
[ROUTER] Strategy  : Warm hybrid (confidence=0.8740)
   1. [0.8839] canon ef 135mm f/2l usm lens for canon slr cameras the ...
   2. [0.8804] nikon 70-200mm f/4g ed vr nikkor zoom lens   nikon...
   3. [0.8779] bluerigger digital optical audio toslink cable (25 feet...
   4. [0.8755] manfrotto 035rl super clamp with 2908 standard stud - r...
   5. [0.8728] nikon 85mm f/1.4g af-s nikkor lens for nikon digital sl...
   6. [0.8710] mobile fidelity - mfsl inner sleeves (pkg 50) the indus...
   7. [0.8708] nikon d300 dx 12.3mp digital slr camera (body only) eng...
   8. [0.8699] canon ef 70-200mm f/2.8l is ii usm telephoto zoom lens ...
   9. [0.8692] nikon d700 12.1mp fx-format cmos digital slr camera wit...
  10. [0.8690] intel core i7-2600k quad-core processor 3.4 ghz 8 mb ca...

Recommendations for new user:
[ROUTER] User tier : new (0 interactions)
[ROUTER] Strategy  : Trending
   1. [1.0000] sandisk 

In [100]:
#similar()
# Find similar products to any item
item_id = item_enc.classes_[0]
title   = item_features[
    item_features['item_id'] == item_id
]['item_text'].values[0][:60]

print(f"Similar products to: {title}...")
print("=" * 55)
similar = ncf_rec.similar(item_id, top_n=10)
for i, r in enumerate(similar, 1):
    print(f"  {i:2}. [{r['similarity']:.4f}] [{r['source']}] {r['title'][:50]}...")

# Try on product page route
print("\nProduct page route:")
print("=" * 55)
page = ncf_router.for_product_page(item_id, top_n=5)
for i, r in enumerate(page, 1):
    print(f"  {i:2}. [{r.get('similarity',0):.4f}] {r['title'][:50]}...")

Similar products to: rand mcnally 528881469 7-inch intelliroute tnd 700 truck gps...
   1. [0.8962] [ALS] new trent icarrier 12000mah portable dual usb port...
   2. [0.8857] [ALS] new trent imirror 6000mah heavy duty 2a/1a dual us...
   3. [0.8832] [ALS] new trent: itorch 5200mah ultra portable usb port ...
   4. [0.8830] [ALS] new trent: idual nt5200 - (5200mah) dual usb exter...
   5. [0.8786] [ALS] new trent gladius ipad case compatible: ipad 4, ip...
   6. [0.8703] [ALS] naxa electronics nab bluetooth wireless receiver a...
   7. [0.8690] [ALS] nextar x3b 3.5-inch bluetooth portable gps navigat...
   8. [0.8681] [ALS] new trent powerpak xtreme 12000mah rugged water/di...
   9. [0.8671] [ALS] new trent chargepak nt600c 6000mah external batter...
  10. [0.8668] [ALS] new trent imp39b/nt-39b embassy keyboard case for ...

Product page route:
[ROUTER] Product page : 0528881469
   1. [0.8962] new trent icarrier 12000mah portable dual usb port...
   2. [0.8857] new trent imirror 6000mah

In [101]:
#trending()
# Global trending
print("Global Trending:")
print("=" * 55)
trend = ncf_rec.trending(top_n=10)
for i, r in enumerate(trend, 1):
    print(f"  {i:2}. [{r['interaction_count']:5} interactions] [{r['avg_rating']}★] {r['title'][:45]}...")

# Category filtered
for category in ['headphone', 'laptop', 'camera', 'cable']:
    print(f"\nTrending — '{category}':")
    print("-" * 55)
    results = ncf_rec.trending(top_n=5, category=category)
    if results:
        for i, r in enumerate(results, 1):
            print(f"  {i}. [{r['interaction_count']:5} interactions] {r['title'][:50]}...")
    else:
        print("  No items found")

Global Trending:
   1. [ 4915 interactions] [4.59★] sandisk ultra 64gb microsdxc class 10 uhs mem...
   2. [ 4143 interactions] [4.8★] amazonbasics high-speed hdmi cable - 15 feet ...
   3. [ 3798 interactions] [4.0★] google chromecast hdmi streaming media player...
   4. [ 3435 interactions] [4.8★] mediabridge ultra series hdmi cable (6 feet) ...
   5. [ 2813 interactions] [4.66★] transcend 8 gb class 10 sdhc flash memory car...
   6. [ 2652 interactions] [4.36★] panasonic rphje120d in-ear headphone, orange ...
   7. [ 2599 interactions] [4.6★] dvi gear hdmi cable 2m 6 feet   dvi gear...
   8. [ 2542 interactions] [4.44★] amazonbasics apple certified lightning to usb...
   9. [ 2104 interactions] [4.42★] roku 3 streaming media player   roku...
  10. [ 2082 interactions] [4.74★] eneloop sec-cspacer4pk c size spacers for use...

Trending — 'headphone':
-------------------------------------------------------
  1. [ 2652 interactions] panasonic rphje120d in-ear headphone, orange panas...


In [ ]:
#You call → ncf_router.for_user(user_id)
              #↓
         #Router checks how many interactions this user has
              #↓
         #Classifies them into a tier
              #↓
         #Calls the right strategy automatically
             # ↓
         #Returns recommendations

In [102]:
# Find users from each tier to demonstrate
interaction_counts = np.diff(train_csr.indptr)

# Pick one user from each tier
new_user_id    = "COMPLETELY_NEW_USER_999"                                          # 0 interactions
cold_user_idx  = np.where((interaction_counts >= 1) & (interaction_counts < 5))[0][0]
warm_user_idx  = np.where((interaction_counts >= 5) & (interaction_counts < 20))[0][0]
active_user_idx= np.where(interaction_counts >= 20)[0][0]

cold_user_id   = user_enc.classes_[cold_user_idx]
warm_user_id   = user_enc.classes_[warm_user_idx]
active_user_id = user_enc.classes_[active_user_idx]

print(f"Cold user   : {cold_user_id}  ({interaction_counts[cold_user_idx]} interactions)")
print(f"Warm user   : {warm_user_id}  ({interaction_counts[warm_user_idx]} interactions)")
print(f"Active user : {active_user_id} ({interaction_counts[active_user_idx]} interactions)")

Cold user   : A000715434M800HLCENK9  (4 interactions)
Warm user   : A00101847G3FJTWYGNQA  (5 interactions)
Active user : A100UD67AHFODS (82 interactions)


In [103]:
#New user → gets trending
print("\n" + "="*55)
print("NEW USER (0 interactions)")
print("="*55)
print("What they get: global trending items")
print("-"*55)

recs = ncf_router.for_user(new_user_id, top_n=5)
for i, r in enumerate(recs, 1):
    print(f"  {i}. [{r.get('score',0):.4f}] [{r.get('strategy')}] {r['title'][:50]}...")


NEW USER (0 interactions)
What they get: global trending items
-------------------------------------------------------
[ROUTER] User tier : new (0 interactions)
[ROUTER] Strategy  : Trending
  1. [1.0000] [trending] sandisk ultra 64gb microsdxc class 10 uhs memory c...
  2. [0.8429] [trending] amazonbasics high-speed hdmi cable - 15 feet ( 4.6...
  3. [0.7727] [trending] google chromecast hdmi streaming media player   go...
  4. [0.6989] [trending] mediabridge ultra series hdmi cable (6 feet) - hig...
  5. [0.5723] [trending] transcend 8 gb class 10 sdhc flash memory card (ts...


In [104]:
#Cold user → NCF + popularity blend
print("\n" + "="*55)
print(f"COLD USER ({interaction_counts[cold_user_idx]} interactions)")
print("="*55)
print("What they get: some personalization + popular items padded in")
print("-"*55)

# Show what they have interacted with
user_items = train_csr[cold_user_idx].indices
print("Their history:")
for idx in user_items[:5]:
    title = item_features[
        item_features['item_id'] == item_enc.classes_[idx]
    ]['item_text'].values
    print(f"  → {title[0][:55] if len(title)>0 else 'unknown'}...")

print("\nRecommendations:")
recs = ncf_router.for_user(cold_user_id, top_n=5)
for i, r in enumerate(recs, 1):
    print(f"  {i}. [{r.get('score',0):.4f}] [{r.get('strategy')}] {r['title'][:50]}...")


COLD USER (4 interactions)
What they get: some personalization + popular items padded in
-------------------------------------------------------
Their history:
  → draper v screen manual projection screen - 70&quot; x 7...
  → mount-it! universal projector wall mount with extendabl...
  → amazonbasics high-speed hdmi cable - 15 feet ( 4.6 mete...
  → gopro case for gopro hero 1/2/3/3+ and accessories - id...

Recommendations:
[ROUTER] User tier : cold (4 interactions)
[ROUTER] Strategy  : Hybrid + 60% popularity blend
  1. [0.8800] [hybrid] canon ef 135mm f/2l usm lens for canon slr cameras...
  2. [0.8759] [hybrid] nikon 70-200mm f/4g ed vr nikkor zoom lens   nikon...
  3. [1.0000] [popularity_blend] sandisk ultra 64gb microsdxc class 10 uhs memory c...
  4. [0.8429] [popularity_blend] amazonbasics high-speed hdmi cable - 15 feet ( 4.6...
  5. [0.7727] [popularity_blend] google chromecast hdmi streaming media player   go...


In [105]:
#Warm user → NCF + confidence check
print("\n" + "="*55)
print(f"WARM USER ({interaction_counts[warm_user_idx]} interactions)")
print("="*55)
print("What they get: NCF personalized, blended only if confidence low")
print("-"*55)

print("Their history:")
user_items = train_csr[warm_user_idx].indices
for idx in user_items[:5]:
    title = item_features[
        item_features['item_id'] == item_enc.classes_[idx]
    ]['item_text'].values
    print(f"  → {title[0][:55] if len(title)>0 else 'unknown'}...")

print("\nRecommendations:")
recs = ncf_router.for_user(warm_user_id, top_n=5)
for i, r in enumerate(recs, 1):
    print(f"  {i}. [{r.get('score',0):.4f}] [{r.get('strategy')}] {r['title'][:50]}...")


WARM USER (5 interactions)
What they get: NCF personalized, blended only if confidence low
-------------------------------------------------------
Their history:
  → thermaltake dr. power ii automated power supply tester ...
  → anker&reg; uspeed usb 3.0 card reader 8-in-1 for sdxc, ...
  → samsung electronics mz-7pd128bw 840 pro series 2.5-inch...
  → asus wireless router (rt-n10p) the rt-n10p wireless-n15...
  → tp-link tl-pa6010kit av600 powerline adapter starter ki...

Recommendations:
[ROUTER] User tier : warm (5 interactions)
[ROUTER] Strategy  : Warm hybrid (confidence=0.9951)
  1. [1.0020] [None] bluerigger digital optical audio toslink cable (25...
  2. [0.9954] [None] nikon 70-200mm f/4g ed vr nikkor zoom lens   nikon...
  3. [0.9945] [None] canon ef 135mm f/2l usm lens for canon slr cameras...
  4. [0.9935] [None] nikon 85mm f/1.4g af-s nikkor lens for nikon digit...
  5. [0.9900] [None] mobile fidelity - mfsl inner sleeves (pkg 50) the ...


In [106]:
#Active user → full NCF hybrid
print("\n" + "="*55)
print(f"ACTIVE USER ({interaction_counts[active_user_idx]} interactions)")
print("="*55)
print("What they get: fully personalized NCF recommendations")
print("-"*55)

print("Their history (first 8 items):")
user_items = train_csr[active_user_idx].indices
for idx in user_items[:8]:
    title = item_features[
        item_features['item_id'] == item_enc.classes_[idx]
    ]['item_text'].values
    print(f"  → {title[0][:55] if len(title)>0 else 'unknown'}...")

print("\nRecommendations:")
recs = ncf_router.for_user(active_user_id, top_n=10)
for i, r in enumerate(recs, 1):
    print(f"  {i:2}. [{r.get('score',0):.4f}] [{r.get('strategy','hybrid')}] {r['title'][:50]}...")


ACTIVE USER (82 interactions)
What they get: fully personalized NCF recommendations
-------------------------------------------------------
Their history (first 8 items):
  → belkin hi-speed usb 2.0 cable (10 feet) connect a usb p...
  → terk indoor am antenna  advantage   terk...
  → sony cdpcx455 400 disc megastorage cd changer (disconti...
  → ge 5-jack adapter (tl26131) 5-jack adaptor, white conne...
  → cisco-linksys wrt54gs wireless-g broadband router with ...
  → c2g / cables to go 03137 18 awg outlet saver power exte...
  → logitech z-2300 thx-certified 2.1 speaker system with s...
  → sennheiser hzr-62 stereo volume control sennheiser head...

Recommendations:
[ROUTER] User tier : active (82 interactions)
[ROUTER] Strategy  : Full hybrid (confidence=0.9931)
   1. [1.0030] [hybrid] bluerigger digital optical audio toslink cable (25...
   2. [0.9976] [hybrid] nikon 70-200mm f/4g ed vr nikkor zoom lens   nikon...
   3. [0.9958] [hybrid] canon ef 135mm f/2l usm lens for canon slr

In [107]:
#Product page → similar items
print("\n" + "="*55)
print("PRODUCT PAGE")
print("="*55)
print("What they get: items similar to what they are viewing")
print("-"*55)

# Simulate user landing on a product page
product_id    = item_enc.classes_[10]
product_title = item_features[
    item_features['item_id'] == product_id
]['item_text'].values[0][:60]

print(f"Viewing: {product_title}...")
print("\nSimilar products:")
similar = ncf_router.for_product_page(product_id, top_n=5)
for i, r in enumerate(similar, 1):
    print(f"  {i}. [{r.get('similarity', r.get('score',0)):.4f}] [{r.get('strategy')}] {r['title'][:50]}...")


PRODUCT PAGE
What they get: items similar to what they are viewing
-------------------------------------------------------
Viewing: nook simple touch ereader the nook simple touch ereader allo...

Similar products:
[ROUTER] Product page : 1400532736
  1. [0.8506] [ALS] hp pavilion m9450f elite desktop pc (2.5 ghz intel...
  2. [0.8344] [ALS] hp envy 17-j029nr quad edition mssd windows 8 note...
  3. [0.8285] [ALS] samsung galaxy tab 2 (7-inch, wi-fi) 2012 model   ...
  4. [0.7938] [ALS] zeikos ze-qc4000 rapid aa/aaa battery charger ac/d...
  5. [0.7926] [ALS]  guards against scratches, smears, dust and dirt, ...


In [108]:
#One line summary of how to get recs for any user
# This is all you ever need to call — router handles everything
def get_recs(user_id, top_n=10):
    return ncf_router.for_user(user_id, top_n=top_n)

def get_similar(item_id, top_n=10):
    return ncf_router.for_product_page(item_id, top_n=top_n)

def get_trending(category=None, top_n=10):
    return ncf_rec.trending(top_n=top_n, category=category)

# Examples
get_recs("ANY_USER_ID")        # auto routes based on their history
get_similar("ANY_ITEM_ID")     # similar products
get_trending(category="camera") # trending cameras

[ROUTER] User tier : new (0 interactions)
[ROUTER] Strategy  : Trending
[ROUTER] Product page : ANY_ITEM_ID


[{'item_id': 'B000QUUFRW',
  'title': 'sandisk 4gb extreme sdhc class 10 memory card the sandisk extreme sdhc',
  'interaction_count': 1890,
  'avg_rating': 4.79,
  'score': 0.3845,
  'source': 'trending'},
 {'item_id': 'B00622AG6S',
  'title': 'powergen 2.4amps / 12w dual usb car charger designed for apple and and',
  'interaction_count': 1710,
  'avg_rating': 4.47,
  'score': 0.3479,
  'source': 'trending'},
 {'item_id': 'B00007E7JU',
  'title': 'canon ef 50mm f/1.8 ii camera lens this is considered the standard len',
  'interaction_count': 1279,
  'avg_rating': 4.59,
  'score': 0.2602,
  'source': 'trending'},
 {'item_id': 'B00004ZCJE',
  'title': 'tiffen 46mm uv protection filter protects lenses from dust, moisture, ',
  'interaction_count': 1258,
  'avg_rating': 4.28,
  'score': 0.256,
  'source': 'trending'},
 {'item_id': 'B004GF8TIK',
  'title': 'mediabridge usb 2.0 - micro-usb to usb cable (6 feet) - high-speed a m',
  'interaction_count': 1204,
  'avg_rating': 4.56,
  'score':

In [89]:
# Proper NCF evaluation
def evaluate_ncf_proper(n_users=1000, k=10):
    test_users  = np.where(np.diff(test_csr.indptr) > 0)[0]
    train_users = np.where(np.diff(train_csr.indptr) > 0)[0]
    valid_users = np.intersect1d(test_users, train_users)
    sample      = np.random.choice(
        valid_users, min(n_users, len(valid_users)), replace=False
    )

    als_p, ncf_p = [], []
    als_r, ncf_r = [], []

    ncf_model.eval()
    for user_idx in sample:
        actual = set(test_csr[user_idx].indices.tolist())
        if not actual:
            continue
        try:
            # ALS
            ids, _ = model.recommend(
                user_idx, train_csr[user_idx],
                N=k, filter_already_liked_items=True
            )
            hits = len(actual & set(ids.tolist()))
            als_p.append(hits / k)
            als_r.append(hits / len(actual))

            # NCF — score ONLY actual test items + 100 random negatives
            # this is standard evaluation practice
            train_items  = set(train_csr[user_idx].indices.tolist())
            test_items   = list(actual)

            # Sample negatives — items user hasn't seen
            all_items    = list(set(range(n_items)) - train_items - actual)
            negatives    = np.random.choice(all_items, min(100, len(all_items)), replace=False)
            candidates   = np.array(test_items + list(negatives))

            with torch.no_grad():
                u      = torch.tensor([user_idx]*len(candidates), dtype=torch.long).to(device)
                v      = torch.tensor(candidates, dtype=torch.long).to(device)
                scores = ncf_model(u, v).cpu().numpy()

            top_k_idx    = candidates[scores.argsort()[::-1][:k]]
            hits_ncf     = len(actual & set(top_k_idx.tolist()))
            ncf_p.append(hits_ncf / k)
            ncf_r.append(hits_ncf / len(actual))

        except Exception:
            continue

    print("=== ALS vs NCF (proper evaluation) ===")
    print(f"ALS  Precision@{k}: {np.mean(als_p):.4f} | Recall@{k}: {np.mean(als_r):.4f}")
    print(f"NCF  Precision@{k}: {np.mean(ncf_p):.4f} | Recall@{k}: {np.mean(ncf_r):.4f}")

    als_wins = np.mean(als_p) > np.mean(ncf_p)
    print(f"\nWinner: {'ALS' if als_wins else 'NCF'} 🏆")

evaluate_ncf_proper()

=== ALS vs NCF (proper evaluation) ===
ALS  Precision@10: 0.0052 | Recall@10: 0.0375
NCF  Precision@10: 0.0271 | Recall@10: 0.1840

Winner: NCF 🏆


In [ ]:
print("Evaluating...")
p_at_10, r_at_10 = precision_at_k(model, train_csr, test_csr, k=10)
print(f"Precision@10 : {p_at_10:.4f}")
print(f"Recall@10    : {r_at_10:.4f}")

In [ ]:
def precision_at_k(model, train_matrix, test_matrix, k=10, n_users=1000):

    n_users_total = train_matrix.shape[0]

    test_users = np.where(np.diff(test_matrix.indptr) > 0)[0]
    train_users = np.where(np.diff(train_matrix.indptr) > 0)[0]
    valid_users = np.intersect1d(test_users, train_users)
    valid_users = valid_users[valid_users < n_users_total]

    sample_users = np.random.choice(
        valid_users,
        min(n_users, len(valid_users)),
        replace=False
    )

    precisions = []
    recalls = []

    for user_idx in sample_users:
        actual = set(test_matrix[user_idx].indices.tolist())
        if not actual:
            continue

        try:
            ids, scores = model.recommend(
                user_idx,
                train_matrix[user_idx],  # ← pass train_csr row directly
                N=k,
                filter_already_liked_items=True
            )
            recommended_items = set(ids.tolist())
            hits = len(actual & recommended_items)
            precisions.append(hits / k)
            recalls.append(hits / len(actual))

        except Exception as e:
            continue

    avg_precision = np.mean(precisions) if precisions else 0.0
    avg_recall = np.mean(recalls) if recalls else 0.0
    return avg_precision, avg_recall

In [ ]:
print("Evaluating...")
p_at_10, r_at_10 = precision_at_k(model, train_csr, test_csr, k=10)
print(f"Precision@10 : {p_at_10:.4f}")
print(f"Recall@10    : {r_at_10:.4f}")

In [ ]:
# Force correct setup — user_factors and item_factors are swapped in this version
# We need to access them correctly

print(f"train_csr shape (user x item)      : {train_csr.shape}")
print(f"train_item_user shape (item x user): {train_item_user.shape}")
print(f"item_factors shape                 : {model.item_factors.shape}")
print(f"user_factors shape                 : {model.user_factors.shape}")

# Check implicit version
print(f"\nimplicit version: {implicit.__version__}")

# Manual recommendation for heavy test user
heavy_test_users = np.where(np.diff(test_csr.indptr) >= 5)[0]
test_user = heavy_test_users[0]

# Try passing train_csr row directly instead of train_item_user row
user_items_row = train_csr[test_user]  # ← direct user row from user-item matrix

ids, scores = model.recommend(
    test_user,
    user_items_row,
    N=10,
    filter_already_liked_items=True
)

actual = set(test_csr[test_user].indices.tolist())
print(f"\nRecommended : {set(ids.tolist())}")
print(f"Actual test : {actual}")
print(f"Hits        : {actual & set(ids.tolist())}")

In [ ]:
import numpy as np

def score(user_id, item_id):
    """
    Score a single user-item pair.
    Returns a float — higher = more relevant.
    """
    # Convert IDs to indices
    try:
        user_idx = np.where(user_enc.classes_ == user_id)[0][0]
        item_idx = np.where(item_enc.classes_ == item_id)[0][0]
    except IndexError:
        return 0.0  # unknown user or item

    # Dot product of user and item embeddings
    user_vec = model.user_factors[user_idx]   # (64,)
    item_vec = model.item_factors[item_idx]   # (64,)

    return float(np.dot(user_vec, item_vec))


def rank_items(user_id, item_ids, top_n=10):
    """
    Given a user and a list of item_ids, rank them by score.
    Useful for re-ranking a candidate set.
    """
    scored = []
    for item_id in item_ids:
        s = score(user_id, item_id)
        scored.append((item_id, s))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_n]


# --- Test it ---
sample_user = user_enc.classes_[active_users[0]]
sample_item = item_enc.classes_[0]

print(f"Score for user={sample_user}, item={sample_item} : {score(sample_user, sample_item):.4f}")

In [ ]:
# Item-Item Similarity
from sklearn.metrics.pairwise import cosine_similarity

# Extract all item embeddings
item_embeddings = model.item_factors  # shape (63001, 64)
print(f"Item embeddings shape: {item_embeddings.shape}")

def get_similar_items(item_id, top_n=10):
    """
    Find top-N most similar items using ALS item embeddings.
    Pure behavior-based similarity — items bought/rated together.
    """
    try:
        item_idx = np.where(item_enc.classes_ == item_id)[0][0]
    except IndexError:
        print(f"Item {item_id} not found")
        return []

    item_vec = item_embeddings[item_idx].reshape(1, -1)  # (1, 64)

    # Compute cosine similarity against all items
    sims = cosine_similarity(item_vec, item_embeddings).flatten()

    # Get top-N (exclude self)
    top_indices = sims.argsort()[::-1][1:top_n+1]

    results = []
    for idx in top_indices:
        similar_item_id = item_enc.classes_[idx]
        title = item_features[
            item_features['item_id'] == similar_item_id
        ]['item_text'].values
        title = title[0][:60] if len(title) > 0 else 'unknown'
        results.append({
            'item_id'   : similar_item_id,
            'title'     : title,
            'similarity': round(float(sims[idx]), 4)
        })

    return results


# --- Test it ---
sample_item_id = item_enc.classes_[0]
print(f"\nItems similar to: {sample_item_id}")
print("-" * 50)
similar = get_similar_items(sample_item_id, top_n=10)
for r in similar:
    print(f"  [{r['similarity']:.4f}] {r['title']}...")

In [ ]:
# Artifact 3 — User Embeddings
user_embeddings = model.user_factors  # shape (192403, 64)
print(f"User embeddings shape: {user_embeddings.shape}")

def get_user_embedding(user_id):
    try:
        user_idx = np.where(user_enc.classes_ == user_id)[0][0]
        return model.user_factors[user_idx]
    except IndexError:
        return None

def get_similar_users(user_id, top_n=5):
    try:
        user_idx = np.where(user_enc.classes_ == user_id)[0][0]
    except IndexError:
        print(f"User {user_id} not found")
        return []

    user_vec = user_embeddings[user_idx].reshape(1, -1)

    sample_idx = np.random.choice(user_embeddings.shape[0], 5000, replace=False)
    sample_vecs = user_embeddings[sample_idx]

    sims = cosine_similarity(user_vec, sample_vecs).flatten()
    top_local = sims.argsort()[::-1][1:top_n+1]
    top_global = sample_idx[top_local]

    results = []
    for idx, local_idx in zip(top_global, top_local):
        results.append({
            'user_id'   : user_enc.classes_[idx],
            'similarity': round(float(sims[local_idx]), 4)
        })

    return results

In [ ]:
# user similarity check
sample_user_id = user_enc.classes_[active_users[0]]
print(f"Similar users to: {sample_user_id}")
print("-" * 50)
similar_users = get_similar_users(sample_user_id)
for u in similar_users:
    print(f"  [{u['similarity']:.4f}] {u['user_id']}")

In [ ]:
print("=== Post-Training Artifacts Summary ===")
print(f"score()             : ✅ shape check — user_factors {model.user_factors.shape}, item_factors {model.item_factors.shape}")
print(f"item_embeddings     : ✅ shape {item_embeddings.shape}")
print(f"user_embeddings     : ✅ shape {user_embeddings.shape}")

# Quick sanity — score should return a float
sample_user = user_enc.classes_[active_users[0]]
sample_item = item_enc.classes_[0]
s = score(sample_user, sample_item)
print(f"score() test        : ✅ {s:.4f}")

# Quick sanity — similar items should return 10 results
sim_items = get_similar_items(item_enc.classes_[0], top_n=10)
print(f"get_similar_items() : ✅ {len(sim_items)} results")

# Quick sanity — similar users should return 5 results
sim_users = get_similar_users(user_enc.classes_[active_users[0]], top_n=5)
print(f"get_similar_users() : ✅ {len(sim_users)} results")

print("\n=== Ready for Hybrid Layer ===")

In [ ]:
#Normalize All Scores to Same Scale
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# --- Normalize item popularity scores (0 to 1) ---
popularity_lookup = item_popularity.set_index('item_id')['interaction_count'].to_dict()

max_pop = max(popularity_lookup.values())
popularity_norm = {k: v / max_pop for k, v in popularity_lookup.items()}

print(f"Popularity lookup ready : {len(popularity_norm)} items")
print(f"Sample popularity score : {list(popularity_norm.items())[0]}")

In [ ]:
# Content Score (TF-IDF based)
from sklearn.metrics.pairwise import cosine_similarity

def get_content_score(user_id, item_id):
    """
    Content score = average TF-IDF similarity between
    target item and items the user has already interacted with.
    """
    try:
        user_idx = np.where(user_enc.classes_ == user_id)[0][0]
        item_idx = np.where(item_enc.classes_ == item_id)[0][0]
    except IndexError:
        return 0.0

    # Items this user has interacted with in training set
    user_train_items = train_csr[user_idx].indices

    if len(user_train_items) == 0:
        return 0.0

    # Limit to 20 items for speed
    sample_items = user_train_items[:20]

    target_vec = tfidf_matrix_ordered[item_idx]
    history_vecs = tfidf_matrix_ordered[sample_items]

    sims = cosine_similarity(target_vec, history_vecs).flatten()
    return float(np.mean(sims))

# Test
print(f"Content score test : {get_content_score(sample_user, sample_item):.4f}")

In [ ]:
#Hybrid Scoring Function
def hybrid_score(user_id, item_id, alpha=0.6, beta=0.3, gamma=0.1):
    """
    Hybrid score combining:
      alpha → ALS collaborative score     (behavior)
      beta  → TF-IDF content score        (content)
      gamma → popularity score            (fallback)

    Weights must sum to 1.0
    """
    # 1. ALS score
    als = score(user_id, item_id)

    # 2. Content score
    content = get_content_score(user_id, item_id)

    # 3. Popularity score
    item_idx = np.where(item_enc.classes_ == item_id)[0]
    pop = popularity_norm.get(item_id, 0.0)

    # Normalize ALS score to 0-1 range (ALS can be negative)
    als_norm = (als + 1) / 2  # rough normalization

    return alpha * als_norm + beta * content + gamma * pop


# Test
h_score = hybrid_score(sample_user, sample_item)
print(f"Hybrid score test : {h_score:.4f}")

In [ ]:
#Full Hybrid Recommendation Function
def hybrid_recommend(user_id, top_n=10, alpha=0.6, beta=0.3, gamma=0.1):
    """
    Generate top-N hybrid recommendations for a user.

    Strategy:
    1. Get top-200 candidates from ALS (fast)
    2. Re-rank using hybrid score (ALS + content + popularity)
    """
    try:
        user_idx = np.where(user_enc.classes_ == user_id)[0][0]
    except IndexError:
        print(f"Unknown user — returning popular items")
        return popularity_fallback(top_n)

    # Step 1 — get 200 candidates from ALS
    ids, als_scores = model.recommend(
        user_idx,
        train_csr[user_idx],
        N=200,
        filter_already_liked_items=True
    )
    candidate_item_ids = [item_enc.classes_[i] for i in ids]

    # Step 2 — re-rank with hybrid score
    scored = []
    for item_id in candidate_item_ids:
        h = hybrid_score(user_id, item_id, alpha, beta, gamma)
        scored.append((item_id, h))

    scored.sort(key=lambda x: x[1], reverse=True)
    top_items = scored[:top_n]

    # Step 3 — format results
    results = []
    for item_id, h in top_items:
        title = item_features[
            item_features['item_id'] == item_id
        ]['item_text'].values
        title = title[0][:70] if len(title) > 0 else 'unknown'
        results.append({
            'item_id' : item_id,
            'title'   : title,
            'score'   : round(h, 4)
        })

    return results


def popularity_fallback(top_n=10):
    """Cold start fallback — return most popular items"""
    top_items = item_popularity.head(top_n)
    results = []
    for _, row in top_items.iterrows():
        title = item_features[
            item_features['item_id'] == row['item_id']
        ]['item_text'].values
        title = title[0][:70] if len(title) > 0 else 'unknown'
        results.append({
            'item_id' : row['item_id'],
            'title'   : title,
            'score'   : round(popularity_norm.get(row['item_id'], 0.0), 4)
        })
    return results

In [ ]:
# Testing
# Test hybrid recommendations
print(f"Hybrid recommendations for: {sample_user}")
print("=" * 60)
recs = hybrid_recommend(sample_user, top_n=10)
for i, r in enumerate(recs, 1):
    print(f"{i:2}. [{r['score']:.4f}] {r['title']}...")

# Test cold start fallback
print(f"\nCold start fallback (unknown user):")
print("=" * 60)
fallback = popularity_fallback(top_n=5)
for i, r in enumerate(fallback, 1):
    print(f"{i:2}. [{r['score']:.4f}] {r['title']}...")

In [ ]:
#Recommend for User
def recommend_for_user(user_id, top_n=10, alpha=0.6, beta=0.3, gamma=0.1):
    """
    Input  : user_id (string)
    Output : top-N personalized recommendations

    Flow:
      Known user   → ALS candidates → hybrid re-rank
      Unknown user → popularity fallback
    """
    user_exists = user_id in user_enc.classes_

    if not user_exists:
        print(f"[INFO] Unknown user '{user_id}' — returning trending items")
        return popularity_fallback(top_n)

    recs = hybrid_recommend(user_id, top_n=top_n, alpha=alpha, beta=beta, gamma=gamma)

    return recs


# --- Test ---
print("=== Service 1: Recommend for User ===")
print(f"\n Known user:")
recs = recommend_for_user(sample_user, top_n=5)
for i, r in enumerate(recs, 1):
    print(f"  {i}. [{r['score']:.4f}] {r['title']}...")

print(f"\n Unknown user (cold start):")
recs_cold = recommend_for_user("UNKNOWN_USER_123", top_n=5)
for i, r in enumerate(recs_cold, 1):
    print(f"  {i}. [{r['score']:.4f}] {r['title']}...")

In [ ]:
# Similar Products
def similar_products(item_id, top_n=10):
    """
    Input  : item_id (string)
    Output : top-N similar items

    Flow:
      Known item   → ALS embedding similarity (primary)
                   → TF-IDF similarity (fallback if low scores)
      Unknown item → TF-IDF similarity only
    """

    item_exists = item_id in item_enc.classes_

    if item_exists:
        # Primary — ALS embedding similarity
        als_similar = get_similar_items(item_id, top_n=top_n)

        # Check if scores are meaningful (> 0.1)
        if als_similar and als_similar[0]['similarity'] > 0.1:
            source = 'ALS embeddings'
            results = als_similar
        else:
            # Fallback — TF-IDF similarity
            source = 'TF-IDF content'
            results = tfidf_similar_products(item_id, top_n=top_n)
    else:
        # Unknown item — TF-IDF only
        source = 'TF-IDF content (unknown item)'
        results = tfidf_similar_products(item_id, top_n=top_n)

    return results, source


def tfidf_similar_products(item_id, top_n=10):
    """TF-IDF based item similarity — content fallback"""
    try:
        # Find item in item_features
        mask = item_features['item_id'] == item_id
        if not mask.any():
            return []

        row_idx = item_features[mask].index[0]

        # Handle index alignment
        if row_idx >= tfidf_matrix_ordered.shape[0]:
            return []

        item_vec = tfidf_matrix_ordered[row_idx]
        sims = cosine_similarity(item_vec, tfidf_matrix_ordered).flatten()
        top_indices = sims.argsort()[::-1][1:top_n+1]

        results = []
        for idx in top_indices:
            sim_item_id = item_features.iloc[idx]['item_id']
            title = item_features.iloc[idx]['item_text'][:70]
            results.append({
                'item_id'   : sim_item_id,
                'title'     : title,
                'similarity': round(float(sims[idx]), 4)
            })
        return results

    except Exception as e:
        print(f"TF-IDF fallback error: {e}")
        return []


# --- Test ---
print("=== Service 2: Similar Products ===")

# Known item
test_item = item_enc.classes_[0]
title = item_features[item_features['item_id'] == test_item]['item_text'].values[0][:60]
print(f"\n Known item: {title}...")

results, source = similar_products(test_item, top_n=5)
print(f" Source: {source}")
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r['similarity']:.4f}] {r['title']}...")

# Unknown item
print(f"\n Unknown item:")
results_unk, source_unk = similar_products("UNKNOWN_ITEM_999", top_n=5)
print(f" Source: {source_unk}")
for i, r in enumerate(results_unk, 1):
    print(f"  {i}. [{r['similarity']:.4f}] {r['title']}...")

In [ ]:
# Trending
def trending(top_n=10, category=None):
    """
    Input  : top_n (int), category (optional string filter)
    Output : top-N trending items by popularity

    Flow:
      No category  → global trending
      With category → filter by category first, then rank
    """

    if category is None:
        # Global trending
        top_items = item_popularity.head(top_n)
    else:
        # Category filtered trending
        category_lower = category.lower()

        # Filter item_features by category keyword in item_text
        mask = item_features['item_text'].str.lower().str.contains(
            category_lower, na=False
        )
        category_items = item_features[mask]['item_id'].tolist()

        if not category_items:
            print(f"[INFO] No items found for category '{category}' — returning global trending")
            top_items = item_popularity.head(top_n)
        else:
            top_items = (
                item_popularity[item_popularity['item_id'].isin(category_items)]
                .head(top_n)
            )

    results = []
    for _, row in top_items.iterrows():
        title = item_features[
            item_features['item_id'] == row['item_id']
        ]['item_text'].values
        title = title[0][:70] if len(title) > 0 else 'unknown'
        results.append({
            'item_id'          : row['item_id'],
            'title'            : title,
            'interaction_count': int(row['interaction_count']),
            'avg_rating'       : round(float(row['avg_rating']), 2),
            'popularity_score' : round(popularity_norm.get(row['item_id'], 0.0), 4)
        })

    return results


# --- Test ---
print("=== Service 3: Trending ===")

print("\n Global trending (top 5):")
global_trend = trending(top_n=5)
for i, r in enumerate(global_trend, 1):
    print(f"  {i}. [{r['interaction_count']} interactions] [{r['avg_rating']}★] {r['title']}...")

print("\n Category trending — 'headphones':")
cat_trend = trending(top_n=5, category='headphone')
for i, r in enumerate(cat_trend, 1):
    print(f"  {i}. [{r['interaction_count']} interactions] [{r['avg_rating']}★] {r['title']}...")

print("\n Category trending — 'laptop':")
laptop_trend = trending(top_n=5, category='laptop')
for i, r in enumerate(laptop_trend, 1):
    print(f"  {i}. [{r['interaction_count']} interactions] [{r['avg_rating']}★] {r['title']}...")

In [ ]:
print("=== Recommender System — All Services Ready ===")
print(f"  Service 1 recommend_for_user() : ✅ personalized + cold start fallback")
print(f"  Service 2 similar_products()   : ✅ ALS primary + TF-IDF fallback")
print(f"  Service 3 trending()           : ✅ global + category filter")
print(f"\n  Model     : ALS (implicit 0.7.2)")
print(f"  Users     : {len(user_enc.classes_):,}")
print(f"  Items     : {len(item_enc.classes_):,}")
print(f"  Interactions : {train_csr.nnz + test_csr.nnz:,}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# ── Helper functions ──────────────────────────────────────────

def get_user_interaction_count(user_id):
    user_idx = np.where(user_enc.classes_ == user_id)[0]
    if len(user_idx) == 0:
        return 0
    return int(train_csr[user_idx[0]].nnz)

def get_recommendation_confidence(user_id, recs):
    if not recs:
        return 0.0
    return np.mean([r['score'] for r in recs])

# ── ElectronicsRecommender class ──────────────────────────────

class ElectronicsRecommender:
    def __init__(self, model, user_enc, item_enc, train_csr,
                 tfidf_matrix, item_features, item_popularity, popularity_norm):
        self.model           = model
        self.user_enc        = user_enc
        self.item_enc        = item_enc
        self.train_csr       = train_csr
        self.tfidf_matrix    = tfidf_matrix
        self.item_features   = item_features
        self.item_popularity = item_popularity
        self.popularity_norm = popularity_norm
        print(f"ElectronicsRecommender ready")
        print(f"  Users : {len(user_enc.classes_):,}")
        print(f"  Items : {len(item_enc.classes_):,}")

    def _user_idx(self, user_id):
        idx = np.where(self.user_enc.classes_ == user_id)[0]
        return idx[0] if len(idx) > 0 else None

    def _item_idx(self, item_id):
        idx = np.where(self.item_enc.classes_ == item_id)[0]
        return idx[0] if len(idx) > 0 else None

    def _item_title(self, item_id):
        row = self.item_features[
            self.item_features['item_id'] == item_id
        ]['item_text'].values
        return row[0][:70] if len(row) > 0 else 'unknown'

    def _popularity_fallback(self, top_n):
        results = []
        for _, row in self.item_popularity.head(top_n).iterrows():
            results.append({
                'item_id' : row['item_id'],
                'title'   : self._item_title(row['item_id']),
                'score'   : round(self.popularity_norm.get(row['item_id'], 0.0), 4),
                'source'  : 'popularity'
            })
        return results

    def _als_score(self, user_idx, item_id):
        item_idx = self._item_idx(item_id)
        if item_idx is None:
            return 0.0
        u = self.model.user_factors[user_idx]
        v = self.model.item_factors[item_idx]
        return float(np.dot(u, v))

    def _content_score(self, user_idx, item_id):
        item_idx = self._item_idx(item_id)
        if item_idx is None:
            return 0.0
        user_items = self.train_csr[user_idx].indices[:20]
        if len(user_items) == 0:
            return 0.0
        target  = self.tfidf_matrix[item_idx]
        history = self.tfidf_matrix[user_items]
        return float(np.mean(cosine_similarity(target, history).flatten()))

    def _hybrid_score(self, user_id, item_id, alpha, beta, gamma):
        user_idx = self._user_idx(user_id)
        if user_idx is None:
            return 0.0
        als     = (self._als_score(user_idx, item_id) + 1) / 2
        content = self._content_score(user_idx, item_id)
        pop     = self.popularity_norm.get(item_id, 0.0)
        return alpha * als + beta * content + gamma * pop

    def recommend(self, user_id, top_n=10, alpha=0.6, beta=0.3, gamma=0.1):
        user_idx = self._user_idx(user_id)
        if user_idx is None:
            return self._popularity_fallback(top_n)
        ids, _ = self.model.recommend(
            user_idx, self.train_csr[user_idx],
            N=200, filter_already_liked_items=True
        )
        candidates = [self.item_enc.classes_[i] for i in ids]
        scored = sorted(
            [(iid, self._hybrid_score(user_id, iid, alpha, beta, gamma))
             for iid in candidates],
            key=lambda x: x[1], reverse=True
        )[:top_n]
        return [{
            'item_id': iid,
            'title'  : self._item_title(iid),
            'score'  : round(s, 4),
            'source' : 'hybrid'
        } for iid, s in scored]

    def similar(self, item_id, top_n=10):
        item_idx = self._item_idx(item_id)
        if item_idx is not None:
            item_vec = self.model.item_factors[item_idx].reshape(1, -1)
            sims     = cosine_similarity(item_vec, self.model.item_factors).flatten()
            top_idx  = sims.argsort()[::-1][1:top_n+1]
            if sims[top_idx[0]] > 0.1:
                return [{
                    'item_id'   : self.item_enc.classes_[i],
                    'title'     : self._item_title(self.item_enc.classes_[i]),
                    'similarity': round(float(sims[i]), 4),
                    'source'    : 'ALS'
                } for i in top_idx]
        row = self.item_features[self.item_features['item_id'] == item_id]
        if row.empty:
            return []
        idx      = row.index[0]
        item_vec = self.tfidf_matrix[idx]
        sims     = cosine_similarity(item_vec, self.tfidf_matrix).flatten()
        top_idx  = sims.argsort()[::-1][1:top_n+1]
        return [{
            'item_id'   : self.item_features.iloc[i]['item_id'],
            'title'     : self.item_features.iloc[i]['item_text'][:70],
            'similarity': round(float(sims[i]), 4),
            'source'    : 'TF-IDF'
        } for i in top_idx]

    def trending(self, top_n=10, category=None):
        if category:
            mask = self.item_features['item_text'].str.lower().str.contains(
                category.lower(), na=False
            )
            items = self.item_features[mask]['item_id'].tolist()
            pool  = self.item_popularity[
                self.item_popularity['item_id'].isin(items)
            ].head(top_n)
        else:
            pool = self.item_popularity.head(top_n)
        return [{
            'item_id'          : row['item_id'],
            'title'            : self._item_title(row['item_id']),
            'interaction_count': int(row['interaction_count']),
            'avg_rating'       : round(float(row['avg_rating']), 2),
            'score'            : round(self.popularity_norm.get(row['item_id'], 0.0), 4),
            'source'           : 'trending'
        } for _, row in pool.iterrows()]

    def score(self, user_id, item_id):
        return round(self._hybrid_score(user_id, item_id, 0.6, 0.3, 0.1), 4)


# ── RecommendationRouter class ────────────────────────────────

class RecommendationRouter:
    NEW_USER_THRESHOLD   = 0
    COLD_USER_THRESHOLD  = 5
    WARM_USER_THRESHOLD  = 20
    CONFIDENCE_THRESHOLD = 0.35

    def __init__(self, recommender):
        self.rec = recommender

    def _classify_user(self, user_id):
        n = get_user_interaction_count(user_id)
        if n == 0:
            return 'new', n
        elif n < self.COLD_USER_THRESHOLD:
            return 'cold', n
        elif n < self.WARM_USER_THRESHOLD:
            return 'warm', n
        else:
            return 'active', n

    def _blend_with_popularity(self, personalized, top_n, blend_ratio=0.4):
        n_popular      = max(1, int(top_n * blend_ratio))
        n_personalized = top_n - n_popular
        popular        = self.rec.trending(top_n=n_popular)
        personalized_ids = {r['item_id'] for r in personalized}
        popular_filtered = [
            r for r in popular
            if r['item_id'] not in personalized_ids
        ][:n_popular]
        for r in personalized[:n_personalized]:
            r['strategy'] = 'hybrid'
        for r in popular_filtered:
            r['strategy'] = 'popularity_blend'
            r['score']    = r.get('score', 0.0)
        return personalized[:n_personalized] + popular_filtered

    def for_user(self, user_id, top_n=10):
        user_tier, n_interactions = self._classify_user(user_id)
        print(f"[ROUTER] User tier : {user_tier} ({n_interactions} interactions)")

        if user_tier == 'new':
            print(f"[ROUTER] Strategy  : Trending (new user)")
            results = self.rec.trending(top_n=top_n)
            for r in results:
                r['strategy'] = 'trending'
            return results

        if user_tier == 'cold':
            print(f"[ROUTER] Strategy  : Hybrid + 60% popularity blend")
            personalized = self.rec.recommend(user_id, top_n=top_n)
            return self._blend_with_popularity(personalized, top_n, blend_ratio=0.6)

        if user_tier == 'warm':
            personalized = self.rec.recommend(user_id, top_n=top_n)
            confidence   = get_recommendation_confidence(user_id, personalized)
            print(f"[ROUTER] Strategy  : Hybrid (confidence={confidence:.4f})")
            if confidence < self.CONFIDENCE_THRESHOLD:
                print(f"[ROUTER] Low confidence → blending with popularity")
                return self._blend_with_popularity(personalized, top_n, blend_ratio=0.3)
            return personalized

        personalized = self.rec.recommend(user_id, top_n=top_n)
        confidence   = get_recommendation_confidence(user_id, personalized)
        print(f"[ROUTER] Strategy  : Full hybrid (confidence={confidence:.4f})")
        if confidence < self.CONFIDENCE_THRESHOLD:
            print(f"[ROUTER] Low confidence → light popularity blend")
            return self._blend_with_popularity(personalized, top_n, blend_ratio=0.2)
        return personalized

    def for_product_page(self, item_id, top_n=10):
        print(f"[ROUTER] Product page : {item_id}")
        similar = self.rec.similar(item_id, top_n=top_n)
        if not similar:
            print(f"[ROUTER] No similar items → popularity fallback")
            results = self.rec.trending(top_n=top_n)
            for r in results:
                r['strategy'] = 'popularity_fallback'
            return results
        source = similar[0].get('source', 'unknown')
        print(f"[ROUTER] Strategy     : {source} similarity")
        if source == 'TF-IDF' and similar[0]['similarity'] < 0.1:
            print(f"[ROUTER] Low TF-IDF scores → blending with popularity")
            n_popular = max(1, top_n // 3)
            popular   = self.rec.trending(top_n=n_popular)
            similar_ids      = {r['item_id'] for r in similar}
            popular_filtered = [
                r for r in popular
                if r['item_id'] not in similar_ids
            ][:n_popular]
            for r in popular_filtered:
                r['strategy'] = 'popularity_blend'
            for r in similar:
                r['strategy'] = source
            return similar[:top_n - n_popular] + popular_filtered
        for r in similar:
            r['strategy'] = source
        return similar

    def explain(self, user_id):
        user_tier, n_interactions = self._classify_user(user_id)
        print(f"\n=== Routing Explanation for {user_id} ===")
        print(f"  Interactions : {n_interactions}")
        print(f"  User tier    : {user_tier}")
        if user_tier == 'new':
            print(f"  Strategy     : Trending (no history)")
        elif user_tier == 'cold':
            print(f"  Strategy     : Hybrid + 60% popularity blend")
        elif user_tier == 'warm':
            print(f"  Strategy     : Hybrid + confidence check")
        else:
            print(f"  Strategy     : Full hybrid")
        print(f"\n  Thresholds:")
        print(f"    New user    : 0 interactions")
        print(f"    Cold user   : 1–{self.COLD_USER_THRESHOLD-1} interactions")
        print(f"    Warm user   : {self.COLD_USER_THRESHOLD}–{self.WARM_USER_THRESHOLD-1} interactions")
        print(f"    Active user : {self.WARM_USER_THRESHOLD}+ interactions")
        print(f"    Confidence  : < {self.CONFIDENCE_THRESHOLD} → blend with popularity")


# ── Instantiate both ──────────────────────────────────────────

rec = ElectronicsRecommender(
    model           = model,
    user_enc        = user_enc,
    item_enc        = item_enc,
    train_csr       = train_csr,
    tfidf_matrix    = tfidf_matrix_ordered,
    item_features   = item_features,
    item_popularity = item_popularity,
    popularity_norm = popularity_norm
)

router = RecommendationRouter(rec)
print("\n✅ rec and router ready")

In [ ]:
# Instantiate recommender first
rec = ElectronicsRecommender(
    model           = model,
    user_enc        = user_enc,
    item_enc        = item_enc,
    train_csr       = train_csr,
    tfidf_matrix    = tfidf_matrix_ordered,
    item_features   = item_features,
    item_popularity = item_popularity,
    popularity_norm = popularity_norm
)

In [ ]:
# Testing routes
router = RecommendationRouter(rec)

# ── Test 1: New user ──────────────────────────────────────────
print("\n" + "="*55)
print("TEST 1 — New User")
print("="*55)
results = router.for_user("BRAND_NEW_USER_999", top_n=5)
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r.get('strategy','?'):20s}] {r['title'][:50]}...")

# ── Test 2: Existing active user ──────────────────────────────
print("\n" + "="*55)
print("TEST 2 — Active User")
print("="*55)
results = router.for_user(sample_user, top_n=5)
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r.get('strategy','?'):20s}] {r['title'][:50]}...")

# ── Test 3: Product page ──────────────────────────────────────
print("\n" + "="*55)
print("TEST 3 — Product Page")
print("="*55)
results = router.for_product_page(item_enc.classes_[0], top_n=5)
for i, r in enumerate(results, 1):
    print(f"  {i}. [{r.get('strategy','?'):20s}] {r['title'][:50]}...")

# ── Test 4: Explain routing ───────────────────────────────────
print("\n" + "="*55)
print("TEST 4 — Explain Routing")
print("="*55)
router.explain(sample_user)
router.explain("BRAND_NEW_USER_999")

In [ ]:
# ── Collect all evaluation metrics ───────────────────────────

from collections import defaultdict
import numpy as np

print("Computing metrics... (takes ~1 min)")

# ── 1. Precision & Recall @ K ────────────────────────────────
def evaluate_at_k(k_values=[5, 10, 20], n_users=1000):
    test_users  = np.where(np.diff(test_csr.indptr) > 0)[0]
    train_users = np.where(np.diff(train_csr.indptr) > 0)[0]
    valid_users = np.intersect1d(test_users, train_users)
    sample      = np.random.choice(valid_users, min(n_users, len(valid_users)), replace=False)

    results = defaultdict(list)

    for user_idx in sample:
        actual = set(test_csr[user_idx].indices.tolist())
        if not actual:
            continue
        try:
            ids, _ = model.recommend(
                user_idx, train_csr[user_idx],
                N=max(k_values), filter_already_liked_items=True
            )
            for k in k_values:
                top_k = set(ids[:k].tolist())
                hits  = len(actual & top_k)
                results[f'precision_{k}'].append(hits / k)
                results[f'recall_{k}'].append(hits / len(actual))
        except:
            continue

    return {key: round(float(np.mean(vals)), 4) for key, vals in results.items()}

metrics = evaluate_at_k(k_values=[5, 10, 20])
print("Precision/Recall done")

# ── 2. Coverage ───────────────────────────────────────────────
def compute_coverage(n_users=500):
    """What % of catalogue appears in recommendations?"""
    test_users  = np.where(np.diff(test_csr.indptr) > 0)[0]
    train_users = np.where(np.diff(train_csr.indptr) > 0)[0]
    valid_users = np.intersect1d(test_users, train_users)
    sample      = np.random.choice(valid_users, min(n_users, len(valid_users)), replace=False)

    recommended_items = set()
    for user_idx in sample:
        try:
            ids, _ = model.recommend(
                user_idx, train_csr[user_idx],
                N=10, filter_already_liked_items=True
            )
            recommended_items.update(ids.tolist())
        except:
            continue

    coverage = len(recommended_items) / len(item_enc.classes_)
    return round(coverage * 100, 2)

metrics['coverage'] = compute_coverage()
print("Coverage done")

# ── 3. User tier distribution ─────────────────────────────────
interaction_counts = np.diff(train_csr.indptr)
tier_counts = {
    'New (0)'      : int((interaction_counts == 0).sum()),
    'Cold (1-4)'   : int(((interaction_counts >= 1) & (interaction_counts < 5)).sum()),
    'Warm (5-19)'  : int(((interaction_counts >= 5) & (interaction_counts < 20)).sum()),
    'Active (20+)' : int((interaction_counts >= 20).sum()),
}
print("Tiers done")

# ── 4. Rating distribution ────────────────────────────────────
rating_counts = {}
for rating in [1.0, 2.0, 3.0, 4.0, 5.0]:
    count = int((interactions['rating'] == rating).sum())
    rating_counts[str(int(rating))] = count
print("Ratings done")

# ── 5. Top 10 most popular items ──────────────────────────────
top10 = item_popularity.head(10).copy()
top10['short_title'] = top10['item_id'].apply(
    lambda x: item_features[
        item_features['item_id'] == x
    ]['item_text'].values[0][:30] if len(
        item_features[item_features['item_id'] == x]
    ) > 0 else x
)
top10_data = top10[['short_title', 'interaction_count', 'avg_rating']].to_dict('records')
print("Top items done")

# ── 6. Interactions per user distribution ────────────────────
bins       = [1, 2, 5, 10, 20, 50, 100, 500]
bin_labels = ['1', '2-4', '5-9', '10-19', '20-49', '50-99', '100+']
bin_counts = []
for i in range(len(bins)-1):
    lo, hi = bins[i], bins[i+1]
    c = int(((interaction_counts >= lo) & (interaction_counts < hi)).sum())
    bin_counts.append(c)

print("\n=== ALL METRICS COLLECTED ===")
print(f"Precision@5  : {metrics['precision_5']}")
print(f"Precision@10 : {metrics['precision_10']}")
print(f"Precision@20 : {metrics['precision_20']}")
print(f"Recall@5     : {metrics['recall_5']}")
print(f"Recall@10    : {metrics['recall_10']}")
print(f"Recall@20    : {metrics['recall_20']}")
print(f"Coverage     : {metrics['coverage']}%")
print(f"Tier counts  : {tier_counts}")